# 5. Supplementary validation and analyses

| | |
|---|---|
| **Stage** | 5 of 9 |
| **Purpose** | Validation work around the main models: metric fixes, review-level leakage and duplicate checks, the blind validation sample for complaint-dominated aspects (S5), data-quality and market audits, the first version analysis and the second dashboard data pack. Sections are tagged S0 to S9. |
| **Inputs** | `data/annotation_v2/gold_set_combined.csv`<br>`outputs/sentiment_audit.csv`<br>`outputs/bert_summary_v2.json`<br>`outputs/predictions/{game}_pairs.parquet` |
| **Outputs** | `outputs/supplementary/s5_* (validation sample, key and annotated files)`<br>`outputs/supplementary/* (evidence tables, version candidates, summaries)`<br>`outputs/dashboard_v2` |
| **Runtime** | GPU required for the Task 3 cross-validation sections; the other sections run on CPU. |
| **Depends on** | Stages 3 and 4 |
| **Configuration** | Set `PROJECT_ROOT` in the first code cell if your Drive folder is not `MyDrive/GameSense`. |

> **Note.** The S5 validation labels feed notebook 07. The dashboard data pack written here is superseded by notebook 09.

In [ ]:
import os, sys, json
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/GameSense'
OUT_DIR      = f'{PROJECT_ROOT}/outputs/supplementary'
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, f'{PROJECT_ROOT}/scripts')
from aspect_taxonomy import ASPECT_TAXONOMY, ASPECT_IDS

GAME_SLUGS   = ["royalmatch", "genshin", "mlbb", "pubgm", "arknights"]
SENT_LABELS  = ["negative", "neutral", "positive"]
L2I          = {l: i for i, l in enumerate(SENT_LABELS)}
GOLD_PATH    = f'{PROJECT_ROOT}/data/annotation_v2/gold_set_combined.csv'
AUDIT_PATH   = f'{PROJECT_ROOT}/outputs/sentiment_audit.csv'
SUMMARY_PATH = f'{PROJECT_ROOT}/outputs/bert_summary_v2.json'

# ---------- Write protection: All write operations to protected directories or files are denied ----------
PROTECTED_DIRS  = [f'{PROJECT_ROOT}/models', f'{PROJECT_ROOT}/data',
                   f'{PROJECT_ROOT}/outputs/predictions', f'{PROJECT_ROOT}/outputs/predictions_v1',
                   f'{PROJECT_ROOT}/outputs/dashboard']
PROTECTED_FILES = [AUDIT_PATH, f'{PROJECT_ROOT}/outputs/deploy_config.json']

def safe_path(p):
    p = os.path.abspath(p)
    assert p not in map(os.path.abspath, PROTECTED_FILES), f"Refused to write to a protected file: {p}"
    for d in PROTECTED_DIRS:
        assert not p.startswith(os.path.abspath(d) + os.sep), f"Refused to write to a protected directory: {p}"
    return p

def update_summary(section, payload):
    """Merge and write the results into the specified section of `bert_summary_v2.json`, while preserving the other content"""
    s = json.load(open(SUMMARY_PATH)) if os.path.exists(SUMMARY_PATH) else {}
    s.setdefault(section, {}).update(payload)
    with open(SUMMARY_PATH, "w") as f:
        json.dump(s, f, indent=2)

print("S0 ready ✅  | New Product Catalog:", OUT_DIR)

S1: Fix up the json file

In [ ]:
# ===== S1: Repair and update bert_summary_v2.json（Task 1 / Task 2）=====
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, average_precision_score

# ---------- 1. Reconstruct the gold label matrix and folds (using the exact same settings as during training)----------
gold_df = pd.read_csv(GOLD_PATH)
Y = np.stack([pd.to_numeric(gold_df[f'gold_{a}'], errors='coerce').fillna(0).astype(int).values
              for a in ASPECT_IDS], axis=1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(np.zeros(len(gold_df)), gold_df['game']))

def macro_ap_fn(y_true, y_prob):
    aps = [average_precision_score(y_true[:, j], y_prob[:, j])
           for j in range(y_true.shape[1]) if y_true[:, j].sum() > 0]
    return float(np.mean(aps))

# ---------- 2. Load saved OOF predictions and check their shapes ----------
oofs = {name: np.load(f'{PROJECT_ROOT}/outputs/oof_{name}.npy')
        for name in ['baseline', 'plain', 'game_prefix']}
for k, v in oofs.items():
    assert v.shape == Y.shape, f"{k} Shape mismatch: {v.shape} vs {Y.shape}"

def cv_row(oof, thr):
    pr = (oof >= thr).astype(int)
    fold_ap = [macro_ap_fn(Y[te], oof[te]) for _, te in folds]
    return {"threshold": thr,
            "macro_ap": round(macro_ap_fn(Y, oof), 3),
            "macro_ap_fold_mean": round(float(np.mean(fold_ap)), 3),
            "macro_ap_fold_sd": round(float(np.std(fold_ap)), 3),
            "macro_f1": round(f1_score(Y, pr, average="macro", zero_division=0), 3),
            "micro_f1": round(f1_score(Y, pr, average="micro", zero_division=0), 3)}

cv = {"tfidf_baseline":   cv_row(oofs['baseline'],    0.5),
      "bert_plain":       cv_row(oofs['plain'],       0.35),
      "bert_game_prefix": cv_row(oofs['game_prefix'], 0.35)}

# ---------- 3. Reproducibility check (should match the results printed during training)----------
expected = {"tfidf_baseline": (0.518, 0.456), "bert_plain": (0.644, 0.618),
            "bert_game_prefix": (0.658, 0.648)}
all_ok = True
for k, (ap, f1) in expected.items():
    ok = abs(cv[k]["macro_ap"] - ap) < 0.002 and abs(cv[k]["macro_f1"] - f1) < 0.002
    all_ok &= ok
    print(f"{k:18s} AP={cv[k]['macro_ap']}  F1={cv[k]['macro_f1']}  {'✅' if ok else '⚠️ Inconsistent with the previous version'}")
assert all_ok, "S1 Reproducibility check failed; writing has stopped."

# ---------- 4. Write to summary (merge section by section; do not overwrite other content) ----------
deploy_thr = json.load(open(f'{PROJECT_ROOT}/outputs/deploy_config.json'))["task2_threshold"]

update_summary("task1_overall_sentiment", {
    "test_macro_f1_bert": 0.664,
    "test_macro_f1_tfidf": 0.631,
    "note": "star-derived weak labels; n_test=5097; recorded from training run"})

update_summary("task2_aspect_detection", {
    "primary_5fold_cv": cv,
    "single_split_test": {"n_test": 75, "threshold": 0.35,
                          "macro_f1": 0.669, "micro_f1": 0.726,
                          "note": "n=75, high variance; recorded from training run"},
    "deployed_threshold": deploy_thr})

update_summary("task3_aspect_sentiment", {
    "v1_gold_only_test_macro_f1": 0.563,
    "v1_note": "v1: gold-only split test, n_test=63, neutral support=1"})

print("\nS1 done ✅ -> bert_summary_v2.json")
print(json.dumps(json.load(open(SUMMARY_PATH)), indent=2))

S2: Task 3 OOF

In [ ]:
# ===== S2: v2 OOF metric =====
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix

# ---------- 1. Reconstruct the data in the exact same order as during training (gold first, then audit）----------
g = pd.read_csv(GOLD_PATH)
rows = []
for _, r in g.iterrows():
    for a in ASPECT_IDS:
        if pd.to_numeric(r.get(f"gold_{a}"), errors="coerce") == 1:
            s = r.get(f"gold_{a}_sentiment")
            if isinstance(s, str) and s in L2I:
                rows.append({"sentence_uid": r["sentence_uid"], "game": r["game"], "aspect": a,
                             "sentence_text": r["sentence_text"], "label": s, "source": "gold"})
gold_pairs = pd.DataFrame(rows)

aud = pd.read_csv(AUDIT_PATH)
aud["human_label"] = aud["human_label"].fillna("").astype(str)
aud = aud[aud["human_label"].isin(SENT_LABELS)].copy()
audit_pairs = aud.rename(columns={"human_label": "label"})[
    ["sentence_uid", "game", "aspect", "sentence_text", "label", "sentiment", "p_negative"]]
audit_pairs["source"] = "audit"

data = pd.concat([gold_pairs, audit_pairs], ignore_index=True)
data["y"] = data["label"].map(L2I)
print(data.groupby(["source", "label"]).size().unstack(fill_value=0))

# ---------- 2. Read OOF and perform alignment checks. ----------
oof = np.load(f'{PROJECT_ROOT}/outputs/task3_oof_group.npy')
assert oof.shape == (len(data), 3), f"Shape mismatch: oof={oof.shape}, data={len(data)}"

y = data["y"].values
pred = oof.argmax(1)
y_neg, pred_neg = (y == 0).astype(int), (pred == 0).astype(int)
aud_m = (data["source"] == "audit").values
gold_m = ~aud_m

# Reproducibility check: Should match the results printed in the Deployment（audit 0.79/0.87，gold 0.832/0.84）
chk = {"audit_acc": (pred_neg[aud_m] == y_neg[aud_m]).mean(),
       "audit_auc": roc_auc_score(y_neg[aud_m], oof[aud_m, 0]),
       "gold_acc":  (pred_neg[gold_m] == y_neg[gold_m]).mean(),
       "gold_auc":  roc_auc_score(y_neg[gold_m], oof[gold_m, 0])}
print({k: round(v, 3) for k, v in chk.items()})
ok = (abs(chk["audit_acc"] - 0.79) < 0.01 and abs(chk["audit_auc"] - 0.87) < 0.01
      and abs(chk["gold_acc"] - 0.832) < 0.005 and abs(chk["gold_auc"] - 0.84) < 0.01)
print("✅ Consistent with the previous version; row order aligned." if ok else "⚠️ It differs from the previous result; please paste the output for me")
assert ok, "S2 Reproducibility check failed; writing has stopped."

# ---------- 3. Indicator function ----------
def metrics(idx):
    yt, yp, po = y[idx], pred[idx], oof[idx]
    f3 = f1_score(yt, yp, average="macro", labels=[0, 1, 2], zero_division=0)
    f_neg_rest = f1_score(yt == 0, yp == 0, average="macro", zero_division=0)
    nn_ = yt != 1                                  # Consider only pairs manually labeled as non-neutral.
    f_neg_pos = (f1_score(yt[nn_] == 0, po[nn_, 0] > po[nn_, 2], average="macro", zero_division=0)
                 if nn_.sum() > 0 and len(set(yt[nn_])) > 1 else np.nan)
    return f3, f_neg_rest, f_neg_pos

def point(mask):
    f3, fnr, fnp = metrics(np.where(mask)[0])
    return {"n": int(mask.sum()),
            "support": {SENT_LABELS[i]: int((y[mask] == i).sum()) for i in range(3)},
            "macro_f1_3class": round(f3, 3),
            "macro_f1_neg_vs_rest": round(fnr, 3),
            "macro_f1_neg_vs_pos_nonneutral": round(fnp, 3)}

res = {"pooled": point(np.ones(len(data), bool)), "gold_only": point(gold_m), "audit_only": point(aud_m)}

# ---------- 4. Bootstrap confidence intervals based on sentence-level clustering (pooled) ----------
rng = np.random.default_rng(42)
grp = list(pd.Series(np.arange(len(data))).groupby(data["sentence_uid"].values).apply(list).values)
boots = []
for _ in range(1000):
    pick = rng.integers(0, len(grp), len(grp))
    boots.append(metrics(np.concatenate([grp[i] for i in pick])))
boots = np.array(boots, dtype=float)
for j, n in enumerate(["macro_f1_3class", "macro_f1_neg_vs_rest", "macro_f1_neg_vs_pos_nonneutral"]):
    lo, hi = np.nanpercentile(boots[:, j], [2.5, 97.5])
    res["pooled"][n + "_ci95"] = [round(float(lo), 3), round(float(hi), 3)]

# ---------- 5. Print ----------
print(json.dumps(res, indent=2))
print("\n=== Pooled per-class report (3 cate) ===")
print(classification_report(y, pred, target_names=SENT_LABELS, zero_division=0, digits=3))
print("Confusion matrix (Rows = Human, Columns = Model; Order: neg/neu/pos):")
print(confusion_matrix(y, pred, labels=[0, 1, 2]))

# ---------- 6. Re-write summary ----------
update_summary("task3_aspect_sentiment", {
    "v2_grouped_5fold_cv": {
        "setup": "gold+audit pairs, StratifiedGroupKFold by sentence_uid, OOF predictions",
        "negative_vs_rest_accuracy": {"audit": round(chk["audit_acc"], 3), "gold": round(chk["gold_acc"], 3)},
        "negative_vs_rest_auc": {"audit": round(chk["audit_auc"], 3), "gold": round(chk["gold_auc"], 3)},
        **res,
        "note": "audit pairs are star/aspect-stratified, not random; gold-only is the cleaner estimate"},
    "note": "v1 = gold-only split test; v2 = gold+audit grouped CV (deployed: task3_final_v2)"})
print("\nS2 done ✅ -> bert_summary_v2.json")

S3: IAA

S3a

In [ ]:
# ===== S3a: Generate IAA blind-labeled samples =====
COMMON = ["performance_stability", "monetization_value", "advertisement",
          "level_difficulty_balance", "content_progression", "paywall", "account_support"]
GENRE = {"royalmatch": [], "genshin": ["gacha_rates", "character_operator_balance"],
         "mlbb": ["character_operator_balance", "matchmaking", "cheating"],
         "pubgm": ["cheating", "matchmaking"],
         "arknights": ["gacha_rates", "character_operator_balance", "grind_progression"]}

N_PER_GAME = 12
g = pd.read_csv(GOLD_PATH)

# For each game, 12 sentences are randomly selected (using a fixed seed for reproducibility) and then shuffled to avoid consecutive annotation by game
parts = [g[g["game"] == s].sample(n=N_PER_GAME, random_state=42) for s in GAME_SLUGS]
samp = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

# ---------- Blind labeling file: Contains no gold labels, weak labels, or model predictions ----------
blind = samp[["sentence_uid", "game", "sentence_text"]].copy()
blind["applicable_aspects"] = blind["game"].map(lambda s: ", ".join(COMMON + GENRE[s]))
for a in ASPECT_IDS:
    blind[f"ann_{a}"] = ""          # Leave blank if the aspect is not mentioned; otherwise, enter negative, neutral, or positive

# ---------- Comparison table (gold label, not shown to the labeler）----------
key_cols = ["sentence_uid", "game", "sentence_text"] + \
           [c for a in ASPECT_IDS for c in (f"gold_{a}", f"gold_{a}_sentiment")]
key = samp[key_cols].copy()

blind_path = safe_path(f"{OUT_DIR}/iaa_blind_for_annotator2.csv")
key_path   = safe_path(f"{OUT_DIR}/iaa_key_gold_DO_NOT_SHARE.csv")
blind.to_csv(blind_path, index=False)
key.to_csv(key_path, index=False)

print(f"Blind bid documents: {blind_path}  ({len(blind)} sentences)")
print(f"Comparison Table:   {key_path}")
print("\nNumber of rounds per game:", blind["game"].value_counts().to_dict())

# Quick check: Distribution of "gold-labeled" aspects in the sample (for your reference only; does not affect the actual annotation)
cnt = {a: int(pd.to_numeric(samp[f"gold_{a}"], errors="coerce").fillna(0).sum()) for a in ASPECT_IDS}
print("sample gold - aspects:", {k: v for k, v in cnt.items() if v > 0})

S3b

In [ ]:
# ===== S3a-2: Rare aspect supplementary samples =====
TARGET_MIN = 5   # The minimum number of positive examples required for each aspect within the set (60 random sentences + supplementary ones).

g = pd.read_csv(GOLD_PATH)
key0 = pd.read_csv(f"{OUT_DIR}/iaa_key_gold_DO_NOT_SHARE.csv")
used = set(key0["sentence_uid"])                  # Sentences already included in the set of 60 random sentences.

def pos_vec(df):
    return np.stack([pd.to_numeric(df[f"gold_{a}"], errors="coerce").fillna(0).astype(int).values
                     for a in ASPECT_IDS], axis=1)

cur = pos_vec(key0).sum(axis=0)                   # Number of positive examples for each aspect across 60 randomly selected sentences.
pool = g[~g["sentence_uid"].isin(used)].reset_index(drop=True)
P = pos_vec(pool)                                  # Label matrix of the candidate pool
need = np.maximum(0, TARGET_MIN - cur)

# ---------- Greedy selection: Prioritize sentences that are scarcer and can simultaneously cover multiple "aspect gaps." ----------
rng = np.random.default_rng(42)
avail = np.ones(len(pool), bool)
picked = []
while need.sum() > 0:
    rem = P[avail].sum(axis=0)                     # How many positive examples remain for each aspect in the candidate pool?
    w = np.where(need > 0, 1.0 / np.maximum(rem, 1), 0.0)
    base = (P * w).sum(axis=1) * avail
    if base.max() <= 0:                            # There are no remaining candidates to fill the vacancies.
        break
    best = int((base + rng.random(len(pool)) * 1e-9).argmax())
    picked.append(best)
    avail[best] = False
    need = np.maximum(0, need - P[best])

top = pool.loc[picked].sample(frac=1, random_state=42).reset_index(drop=True)   # Shuffle the order

# ---------- Output file (format is exactly the same as S3a）----------
blind2 = top[["sentence_uid", "game", "sentence_text"]].copy()
blind2["applicable_aspects"] = blind2["game"].map(lambda s: ", ".join(COMMON + GENRE[s]))
for a in ASPECT_IDS:
    blind2[f"ann_{a}"] = ""

key_cols = ["sentence_uid", "game", "sentence_text"] + \
           [c for a in ASPECT_IDS for c in (f"gold_{a}", f"gold_{a}_sentiment")]
key2 = top[key_cols].copy()

b2 = safe_path(f"{OUT_DIR}/iaa_blind_topup_for_annotator2.csv")
k2 = safe_path(f"{OUT_DIR}/iaa_key_topup_gold_DO_NOT_SHARE.csv")
blind2.to_csv(b2, index=False); key2.to_csv(k2, index=False)

# ---------- Summary ----------
comb = cur + pos_vec(top).sum(axis=0)
summary = pd.DataFrame({"random60": cur, "topup": pos_vec(top).sum(axis=0), "combined": comb,
                        "gold_total": pos_vec(g).sum(axis=0)}, index=ASPECT_IDS)
print(f"Supplementary samples: {len(top)} sentences -> {b2}")
print("Number of sentences per game for the supplementary sample:", top["game"].value_counts().to_dict())
print("\neach aspect Number of positive examples:")
print(summary.to_string())
short = [a for a, c in zip(ASPECT_IDS, comb) if c < TARGET_MIN and pos_vec(g).sum(axis=0)[ASPECT_IDS.index(a)] > 0]
print("\nUnable to reach the target (limited by the total amount of gold):", short)

S4: Leak check for 'gold' reviewId

S4a

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
import re

def rid(uid):                      # sentence_uid = reviewId + "_" + sentence_idx
    return str(uid).rsplit("_", 1)[0]

def norm(t):                       # Text normalization for detecting duplicate sentences.
    return re.sub(r"\s+", " ", str(t).lower()).strip()

gd = gold_df.copy()
gd["reviewId"] = gd["sentence_uid"].map(rid)
gd["norm"] = gd["sentence_text"].map(norm)
rids, norms = gd["reviewId"].values, gd["norm"].values

# ---------- A. gold's comment – ​​sentence structure ----------
per_rev = gd.groupby("reviewId").size()
multi = per_rev[per_rev >= 2]
A = {"n_sentences": int(len(gd)), "n_unique_reviews": int(gd["reviewId"].nunique()),
     "reviews_with_2plus_sentences": int(len(multi)),
     "sentences_in_multi_sentence_reviews": int(multi.sum())}
print("A. gold structure:", A)

# ---------- B. Leak statistics function ----------
def leak_stats(splits, name, keys=rids):
    n_te = n_lk = 0
    per_fold = []
    for tr, te in splits:
        tr_set = set(keys[tr])
        lk = sum(k in tr_set for k in keys[te])
        per_fold.append(int(lk)); n_te += len(te); n_lk += lk
    print(f"  {name:38s} test samples {n_te:4d} | shared with the training set {n_lk:3d} ({100*n_lk/n_te:.1f}%) | each fold: {per_fold}")
    return {"n_test": int(n_te), "n_leaked": int(n_lk), "pct": round(100 * n_lk / n_te, 1)}

# ---------- C. Segmentation for Task 2 ----------
idx = np.arange(len(gd))
idx_train, idx_temp = train_test_split(idx, test_size=0.30, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=42)

print("\nB. The same reviewId appears in both training and testing (reviewId leakage）")
C = {"task2_single_split_test": leak_stats([(idx_train, idx_test)], "Task2 Single segmentation (test)"),
     "task2_single_split_val":  leak_stats([(idx_train, idx_val)],  "Task2 Single segmentation (val)"),
     "task2_5fold_cv":          leak_stats(folds, "Task2 5-fold CV")}

print("\nC. Sentences with identical text appear in both the training and testing sets.")
D = {"task2_single_split_test": leak_stats([(idx_train, idx_test)], "Task2 Single segmentation (test)", norms),
     "task2_5fold_cv":          leak_stats(folds, "Task2 5-fold CV", norms)}

# ---------- D. Task 3 grouping CV (grouped by sentence_uid, requiring the same splitting as during training）----------
d3 = data.copy()
d3["reviewId"] = d3["sentence_uid"].map(rid)
d3["norm"] = d3["sentence_text"].map(norm)
sp = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
splits3 = list(sp.split(d3, d3["y"], groups=d3["sentence_uid"]))
print("\nD. Task 3 group CV (audit + gold pairs)")
E = {"task3_grouped_cv_by_review":  leak_stats(splits3, "Task3 CV：reviewId leak", d3["reviewId"].values),
     "task3_grouped_cv_by_text":    leak_stats(splits3, "Task3 CV：Repeated text", d3["norm"].values)}

# gDo 'old' and 'audit' share comments?
g_rev = set(d3.loc[d3["source"] == "gold", "reviewId"])
a_rev = set(d3.loc[d3["source"] == "audit", "reviewId"])
E["gold_audit_shared_reviews"] = int(len(g_rev & a_rev))
print(f"  Number of comments shared between gold and audit: {len(g_rev & a_rev)}  (gold {len(g_rev)} , audit {len(a_rev)} )")

# ---------- E. Saved ----------
checks = {"gold_structure": A, "reviewId_leak": C, "duplicate_text_leak": D, "task3": E}
update_summary("data_split_checks", checks)
print("\nS4 done ✅ -> bert_summary_v2.json['data_split_checks']")

S4b

In [ ]:
# ===== S4b: Sensitivity analysis excluding leakage samples=====
def leaked_mask(splits, keys, n):
    m = np.zeros(n, bool)
    for tr, te in splits:
        trs = set(keys[tr])
        for i in te:
            if keys[i] in trs:
                m[i] = True
    return m

# ---------- Task 2：5 fold CV ----------
m2 = leaked_mask(folds, rids, len(gd))
keep2 = ~m2
print(f"Task 2: delet {int(m2.sum())} ，Remaining {int(keep2.sum())} ")
rows2 = {}
for name, oof_, thr in [("tfidf_baseline", oofs["baseline"], 0.5),
                        ("bert_plain", oofs["plain"], 0.35),
                        ("bert_game_prefix", oofs["game_prefix"], 0.35)]:
    full_ap, full_f1 = macro_ap_fn(Y, oof_), f1_score(Y, (oof_ >= thr).astype(int), average="macro", zero_division=0)
    cl_ap = macro_ap_fn(Y[keep2], oof_[keep2])
    cl_f1 = f1_score(Y[keep2], (oof_[keep2] >= thr).astype(int), average="macro", zero_division=0)
    rows2[name] = {"macro_ap_full": round(full_ap, 3), "macro_ap_clean": round(cl_ap, 3),
                   "macro_f1_full": round(full_f1, 3), "macro_f1_clean": round(cl_f1, 3)}
print(pd.DataFrame(rows2).T.to_string())

# ---------- Task 3：group CV ----------
m3 = leaked_mask(splits3, d3["reviewId"].values, len(d3))
keep3 = np.where(~m3)[0]
print(f"\nTask 3: delet {int(m3.sum())} pairs，remaining {len(keep3)} ")
f3_full, fnr_full, _ = metrics(np.arange(len(d3)))
f3_cl, fnr_cl, _ = metrics(keep3)
rows3 = {"3class_macro_f1": (round(f3_full, 3), round(f3_cl, 3)),
         "neg_vs_rest_macro_f1": (round(fnr_full, 3), round(fnr_cl, 3))}
print(pd.DataFrame(rows3, index=["full", "clean"]).T.to_string())

update_summary("data_split_checks", {
    "sensitivity_excluding_leaked": {
        "task2_5fold_cv": rows2,
        "task3_grouped_cv": {k: {"full": v[0], "clean": v[1]} for k, v in rows3.items()}}})
print("\nS4b done ✅")

Gold sentences were drawn from 496 distinct reviews (500 sentences); review-level overlap between training and test folds affected at most 1.6% of test items, and excluding them changed all reported metrics by ≤0.01.

S5

S5a: Affective validation of the complaint-dominant aspect

In [ ]:
import re

ASPECTS_S5   = ["performance_stability", "matchmaking", "cheating", "account_support", "advertisement"]
N_PER_ASPECT = 30        # can be changed as 15~20
MIN_P_ASPECT = 0.5
SEED         = 42

CUES = {
 "performance_stability": r"\b(smooth|smoothly|no lag|lag[- ]?free|without lag|runs? (well|great|fine|perfectly)|stable|optimi[sz]ed|no crash(es)?|never crash(es)?|no bugs?|bug[- ]?free|no issues?|works? (well|great|perfectly|fine)|loads? fast|fast loading)\b",
 "matchmaking": r"\b(fair|balanced|well[- ]balanced|even match(es)?|(good|great) match(es|making)?|match ?making (is|was|has been) (good|great|fair|fine|better|improved)|improved match ?making)\b",
 "cheating": r"\b(no (cheaters?|hackers?|cheating|hacks?)|cheat[- ]?free|hacker[- ]?free|(good|great) anti[- ]?cheat|anti[- ]?cheat (is|works|has)|fair play|rarely (see|meet|encounter) (cheaters?|hackers?))\b",
 "account_support": r"\b(great|good|excellent|amazing|fast|quick|helpful|friendly|responsive) (customer )?(support|service|response|team|staff)\b|\b(support|customer service|team|developers?) (is|are|was|were) (great|good|helpful|responsive|quick|fast)\b|\b(resolved|recovered|restored|fixed) (my|the) (account|issue|problem)\b|\beasy to (recover|restore)\b|\bquickly (resolved|replied|responded)\b",
 "advertisement": r"\b(no ads?|ad[- ]?free|few ads?|not (too )?many ads?|(ads?|adverts?) (are|is) (fine|ok|okay|optional|not (bad|annoying|intrusive|too bad))|optional ads?|rewarded ads?|not intrusive|without ads?|doesn.?t (have|show) (any )?ads?)\b",
}

# ---------- 1. Read the full set of pair predictions and construct a sampling frame ----------
pairs = pd.concat([pd.read_parquet(f'{PROJECT_ROOT}/outputs/predictions/{s}_pairs.parquet')
                   for s in GAME_SLUGS], ignore_index=True)
need = ["sentence_uid", "reviewId", "game", "sentence_text", "review_score",
        "aspect", "p_aspect", "p_negative", "sentiment"]
miss = [c for c in need if c not in pairs.columns]
assert not miss, f"pairs missing column: {miss}"
pairs["stars"] = pd.to_numeric(pairs["review_score"], errors="coerce")

excl = set(pd.read_csv(GOLD_PATH)["sentence_uid"]) | set(pd.read_csv(AUDIT_PATH)["sentence_uid"])
frame = pairs[pairs["aspect"].isin(ASPECTS_S5) & (pairs["p_aspect"] >= MIN_P_ASPECT)
              & ~pairs["sentence_uid"].isin(excl)].copy()
print(f"sampling frame: {len(frame)} pair（Sentences labeled gold and those from the previous audit have been excluded）")

# ---------- 2. Stratified sampling ----------
n_cue = int(round(0.33 * N_PER_ASPECT))
n_hi  = int(round(0.40 * N_PER_ASPECT))
n_lo  = N_PER_ASPECT - n_cue - n_hi
rng = np.random.default_rng(SEED)
used_uid, chosen, log = set(), [], []

def take(df, n, stratum):
    df = df[~df["sentence_uid"].isin(used_uid)]
    k = min(n, len(df))
    s = df.sample(n=k, random_state=int(rng.integers(0, 10**9))) if k > 0 else df.head(0)
    used_uid.update(s["sentence_uid"])
    return s.assign(stratum=stratum), k

for a in ASPECTS_S5:
    fa = frame[frame["aspect"] == a]
    cue_pool = fa[fa["sentence_text"].str.contains(CUES[a], flags=re.I, regex=True, na=False)]
    hi_pool, lo_pool = fa[fa["stars"] >= 4], fa[fa["stars"] <= 2]
    s_cue, k_cue = take(cue_pool, n_cue, "cue_positive")
    s_hi,  k_hi  = take(hi_pool, n_hi + (n_cue - k_cue), "hi_star_random")   # Use "hi" to supplement when the cue is insufficient.
    s_lo,  k_lo  = take(lo_pool, n_lo, "lo_star_random")
    chosen += [s_cue, s_hi, s_lo]
    log.append({"aspect": a, "cue_pool": len(cue_pool), "hi_pool": len(hi_pool), "lo_pool": len(lo_pool),
                "picked_cue": k_cue, "picked_hi": k_hi, "picked_lo": k_lo})
print(pd.DataFrame(log).to_string(index=False))

# ---------- 3.Sort by "round": select one item for each aspect-stratum combination per round; the stopping point ensures a balanced distribution ----------
sel = pd.concat(chosen, ignore_index=True)
sel["rank_in_stratum"] = sel.groupby(["aspect", "stratum"]).cumcount()
sel = sel.sort_values(["rank_in_stratum", "aspect", "stratum"]).reset_index(drop=True)
sel["item_id"] = np.arange(1, len(sel) + 1)

# ---------- 4. Output: Blind labeling file (without model predictions) + Reference table ----------
blind = pd.DataFrame({
    "item_id": sel["item_id"], "aspect": sel["aspect"],
    "aspect_label": sel["aspect"].map(lambda a: ASPECT_TAXONOMY[a]["label"]),
    "aspect_description": sel["aspect"].map(lambda a: ASPECT_TAXONOMY[a]["description"]),
    "sentence_text": sel["sentence_text"], "human_label": ""})
key = sel[["item_id", "sentence_uid", "reviewId", "game", "aspect", "stratum", "stars",
           "p_aspect", "p_negative", "sentiment"]].rename(columns={"sentiment": "model_sentiment"})

bp = safe_path(f"{OUT_DIR}/s5_blind_for_annotation.csv")
kp = safe_path(f"{OUT_DIR}/s5_key_DO_NOT_OPEN_BEFORE_ANNOTATING.csv")
blind.to_csv(bp, index=False); key.to_csv(kp, index=False)
print(f"\nBlind annotated documents: {bp}  ({len(blind)} )\nComparison Table:   {kp}")
print("stratum:", sel["stratum"].value_counts().to_dict())

In [ ]:
# ===== S5 annotation tool (needs S0: OUT_DIR, ASPECT_TAXONOMY, safe_path) =====
import html, shutil
import ipywidgets as widgets
from IPython.display import display

BLIND_PATH = f"{OUT_DIR}/s5_blind_for_annotation.csv"
WORK_PATH  = f"{OUT_DIR}/s5_annotated.csv"      # working file: auto-saved after every click

def annotate_s5():
    # Resume from the working file if it exists, otherwise start from the blind file
    if not os.path.exists(WORK_PATH):
        shutil.copy(BLIND_PATH, safe_path(WORK_PATH))
        print("Created working file:", WORK_PATH)
    df = pd.read_csv(WORK_PATH)
    df["human_label"] = df["human_label"].fillna("").astype(str).str.strip().str.lower()

    state = {"i": None, "hist": []}
    head, body = widgets.HTML(), widgets.HTML()
    rules = widgets.HTML(
        "<div style='font-size:12px;color:#999;margin-top:8px'>"
        "Judge the attitude toward the <b>aspect</b>, not the whole sentence. "
        "Use <b>na</b> if the aspect is not actually discussed "
        "(also for pure emotion, feature requests, truncated fragments)."
        "</div>")

    def pending():
        return df.index[df["human_label"] == ""].tolist()

    def show():
        t = pending()
        done = len(df) - len(t)
        counts = df["human_label"].value_counts().to_dict()
        counts.pop("", None)
        if not t:
            head.value = f"<h3>All done ({done}/{len(df)})</h3><span style='color:#999'>{counts}</span>"
            body.value, state["i"] = "", None
            return
        i = t[0]
        state["i"] = i
        r = df.loc[i]
        meta = ASPECT_TAXONOMY[r["aspect"]]
        head.value = (f"<b>Progress {done}/{len(df)}</b> &nbsp; "
                      f"<span style='font-size:12px;color:#999'>{counts}</span>")
        body.value = (f"<p style='font-size:19px'>{html.escape(str(r['sentence_text']))}</p>"
                      f"<p style='color:#8fd0ff'>Aspect: <b>{html.escape(meta['label'])}</b><br>"
                      f"<span style='font-size:12px;color:#aaa'>{html.escape(meta['description'])}</span></p>")

    def save():
        df.to_csv(safe_path(WORK_PATH), index=False)

    def set_label(v):
        if state["i"] is None:
            return
        df.at[state["i"], "human_label"] = v
        state["hist"].append(state["i"])
        save()
        show()

    def undo(_):
        if not state["hist"]:
            return
        i = state["hist"].pop()
        df.at[i, "human_label"] = ""
        save()
        show()

    buttons = []
    for v, style, text in [("negative", "danger", "negative"),
                           ("neutral", "", "neutral"),
                           ("positive", "success", "positive"),
                           ("na", "warning", "na (aspect not discussed)")]:
        b = widgets.Button(description=text, button_style=style, layout=widgets.Layout(width="200px"))
        b.on_click(lambda _, v=v: set_label(v))
        buttons.append(b)
    u = widgets.Button(description="Undo last", layout=widgets.Layout(width="140px"))
    u.on_click(undo)
    buttons.append(u)

    show()
    display(widgets.VBox([head, body, widgets.HBox(buttons), rules]))

annotate_s5()

S5b: Evaluate code

In [ ]:
# ===== S5b: Evaluate model sentiment on complaint-dominant aspects (CPU only; needs S0) =====
# Works on partial annotation: only rows with a non-empty human_label are used.
from scipy.stats import beta

ANNOT_PATH = f"{OUT_DIR}/s5_annotated.csv"
KEY_PATH   = f"{OUT_DIR}/s5_key_DO_NOT_OPEN_BEFORE_ANNOTATING.csv"
VALID      = {"negative", "neutral", "positive", "na"}
PREC_OK    = 0.85   # decision threshold for negative precision in 4-5 star items

ann = pd.read_csv(ANNOT_PATH)
key = pd.read_csv(KEY_PATH)

# ---------- 1. Clean labels and merge with the key ----------
ann["human_label"] = ann["human_label"].fillna("").astype(str).str.strip().str.lower()
bad = ann[(ann["human_label"] != "") & (~ann["human_label"].isin(VALID))]
if len(bad):
    print(f"WARNING: {len(bad)} rows have invalid labels and are ignored. item_id:", bad["item_id"].tolist()[:20])
ann = ann[ann["human_label"].isin(VALID)][["item_id", "human_label"]]
df = key.merge(ann, on="item_id", how="inner")
print(f"Annotated items used: {len(df)} / {len(key)}")
assert len(df) > 0, "No annotated rows found."

df["model_neg"] = (df["model_sentiment"] == "negative").astype(int)
df["valid"]     = df["human_label"] != "na"          # 'na' = aspect not actually discussed (Task 2 false positive)
df["human_neg"] = (df["human_label"] == "negative").astype(int)

def wilson(k, n, a=0.05):
    if n == 0:
        return (np.nan, np.nan)
    lo = 0.0 if k == 0 else beta.ppf(a / 2, k, n - k + 1)
    hi = 1.0 if k == n else beta.ppf(1 - a / 2, k + 1, n - k)
    return (lo, hi)   # Clopper-Pearson (exact) interval; slightly conservative for small n

def prop(k, n):
    lo, hi = wilson(k, n)
    return {"k": int(k), "n": int(n), "p": round(k / n, 3) if n else np.nan,
            "ci95": [round(lo, 3), round(hi, 3)] if n else [np.nan, np.nan]}

# ---------- 2. Per aspect x stratum table ----------
rows = []
for (a, s), d in df.groupby(["aspect", "stratum"]):
    v = d[d["valid"]]
    mn = v[v["model_neg"] == 1]                        # items the model calls negative
    rows.append({"aspect": a, "stratum": s, "n_labeled": len(d),
                 "na_rate": round(1 - len(v) / len(d), 2),
                 "n_valid": len(v),
                 "human_neg_rate": round(v["human_neg"].mean(), 2) if len(v) else np.nan,
                 "model_neg_rate": round(v["model_neg"].mean(), 2) if len(v) else np.nan,
                 "neg_precision": round(mn["human_neg"].mean(), 2) if len(mn) else np.nan,
                 "n_model_neg": len(mn)})
tab = pd.DataFrame(rows)
print("\n=== Per aspect x stratum ===")
print(tab.to_string(index=False))

# ---------- 3. Per aspect summary and decision ----------
summ, out = [], {}
for a in sorted(df["aspect"].unique()):
    da = df[(df["aspect"] == a)]
    va = da[da["valid"]]
    hi = va[(va["stratum"] == "hi_star_random") & (va["model_neg"] == 1)]
    lo = va[(va["stratum"] == "lo_star_random") & (va["model_neg"] == 1)]
    hp = va[va["human_label"] == "positive"]
    p_hi, p_lo = prop(hi["human_neg"].sum(), len(hi)), prop(lo["human_neg"].sum(), len(lo))
    r_pos = prop((hp["model_neg"] == 0).sum(), len(hp))   # positive recall: human-positive that model did not call negative
    if p_hi["n"] < 5:
        verdict = "insufficient 4-5 star items"
    elif p_hi["p"] >= PREC_OK:
        verdict = "complaint-dominant CONFIRMED"
    else:
        verdict = "REVIEW (model may over-predict negative)"
    summ.append({"aspect": a, "n_labeled": len(da), "aspect_precision(non-na)": round(len(va) / len(da), 2),
                 "negprec_4-5star": f'{p_hi["k"]}/{p_hi["n"]}={p_hi["p"]}', "ci95": p_hi["ci95"],
                 "negprec_1-2star": f'{p_lo["k"]}/{p_lo["n"]}={p_lo["p"]}',
                 "pos_recall": f'{r_pos["k"]}/{r_pos["n"]}', "verdict": verdict})
    out[a] = {"n_labeled": int(len(da)), "aspect_precision_non_na": round(len(va) / len(da), 3),
              "neg_precision_hi_star": p_hi, "neg_precision_lo_star": p_lo,
              "positive_recall": r_pos, "n_human_positive": int(len(hp)), "verdict": verdict}
print("\n=== Summary and decision (neg_precision threshold = %.2f) ===" % PREC_OK)
print(pd.DataFrame(summ).to_string(index=False))

# ---------- 4. Overall label distribution and save ----------
print("\nHuman label distribution:", df["human_label"].value_counts().to_dict())
tab.to_csv(safe_path(f"{OUT_DIR}/s5_results_by_stratum.csv"), index=False)
update_summary("s5_complaint_dominant_check", {
    "n_annotated": int(len(df)), "n_total": int(len(key)),
    "human_label_distribution": df["human_label"].value_counts().to_dict(),
    "per_aspect": out,
    "note": "random 4-5 star and 1-2 star strata reported separately; cue stratum is targeted, used only for positive recall"})
print("\nS5b done -> bert_summary_v2.json['s5_complaint_dominant_check']")

S5c

In [ ]:
# ===== S5c: Inspect model-vs-human disagreements (CPU only; needs S0) =====
ann = pd.read_csv(f"{OUT_DIR}/s5_annotated.csv")
key = pd.read_csv(f"{OUT_DIR}/s5_key_DO_NOT_OPEN_BEFORE_ANNOTATING.csv")
df = key.merge(ann[["item_id", "sentence_text", "human_label"]], on="item_id")
df["human_label"] = df["human_label"].fillna("").astype(str).str.strip().str.lower()

def show(d, title, n=12):
    print(f"\n===== {title} ({len(d)}) =====")
    for _, r in d.sort_values("p_negative", ascending=False).head(n).iterrows():
        print(f"[{r['human_label']:8s}] stars={r['stars']} p_neg={r['p_negative']:.2f} | {str(r['sentence_text'])[:150]}")

# 1) Model says negative, human says positive/neutral (the over-prediction cases)
over = df[(df["model_sentiment"] == "negative") & df["human_label"].isin(["positive", "neutral"])]
print("Over-predicted negative, by aspect:", over.groupby("aspect").size().to_dict())
for a in ["advertisement", "account_support", "cheating"]:
    show(over[over["aspect"] == a], f"{a}: model=negative, human=positive/neutral")

# 2) Human says negative, model says non-negative (the opposite error)
under = df[(df["model_sentiment"] != "negative") & (df["human_label"] == "negative")]
show(under, "model=non-negative, human=negative")

# 3) Save all disagreements for error analysis in the report
dis = df[(df["human_label"].isin(["negative", "neutral", "positive"])) &
         ((df["model_sentiment"] == "negative") != (df["human_label"] == "negative"))]
dis.to_csv(safe_path(f"{OUT_DIR}/s5_disagreements.csv"), index=False)
print(f"\nSaved {len(dis)} disagreements -> s5_disagreements.csv")

Apply the corrected code

In [ ]:
# ===== S5-fix: Apply reviewed label corrections to s5_annotated.csv (CPU only; needs S0) =====
# 1) Keep DRY_RUN = True first to preview. 2) Edit CORRECTIONS as you see fit. 3) Set DRY_RUN = False to apply.
DRY_RUN = False

# (aspect, unique substring of the sentence, new label, reason)
CORRECTIONS = [
    ("advertisement",   "minus the ads",                         "negative", "ads are the one disliked part"),
    ("account_support", "after getting in contact with customer service", "positive", "support resolved the issue"),
    ("account_support", "Please choose my account to BETA",      "na",       "request, not a support discussion"),
    ("account_support", "smoothest running",                     "na",       "performance praise, not account/support"),
    ("cheating",        "The appeal should have option",         "na",       "feature suggestion"),
    ("cheating",        "increase my MMR",                       "na",       "MMR/matchmaking, not cheating"),
    ("cheating",        "without any cheat gameplay",            "positive", "absence of cheating praised (cf. 'no ads')"),
]

WORK = f"{OUT_DIR}/s5_annotated.csv"
BACKUP = f"{OUT_DIR}/s5_annotated_before_corrections.csv"
ann = pd.read_csv(WORK)
ann["human_label"] = ann["human_label"].fillna("").astype(str).str.strip().str.lower()

log = []
for aspect, sub, new, why in CORRECTIONS:
    m = ann[(ann["aspect"] == aspect) & ann["sentence_text"].astype(str).str.contains(sub, case=False, regex=False)]
    if len(m) != 1:
        print(f"SKIP ({len(m)} matches): [{aspect}] '{sub}'")
        continue
    i = m.index[0]
    old = ann.at[i, "human_label"]
    if old == new:
        print(f"already '{new}': [{aspect}] '{sub}'")
        continue
    log.append({"item_id": ann.at[i, "item_id"], "aspect": aspect, "old": old, "new": new, "reason": why,
                "sentence_text": ann.at[i, "sentence_text"]})
    ann.at[i, "human_label"] = new

log = pd.DataFrame(log)
print(f"\n{len(log)} corrections {'(preview only)' if DRY_RUN else 'applied'}:")
if len(log):
    print(log[["item_id", "aspect", "old", "new", "reason"]].to_string(index=False))

if not DRY_RUN and len(log):
    if not os.path.exists(BACKUP):                      # keep the pre-correction file once, for transparency
        pd.read_csv(WORK).to_csv(safe_path(BACKUP), index=False)
    ann.to_csv(safe_path(WORK), index=False)
    log.to_csv(safe_path(f"{OUT_DIR}/s5_label_corrections_log.csv"), index=False)
    print("Saved: s5_annotated.csv (updated), backup + corrections log written.")

Check advertisement and account_support.

In [ ]:
# ===== S5d: Blind verification file for annotator 2 (advertisement + account_support) =====
# CPU only; needs S0. Contains NO annotator-1 labels, strata, or model predictions.
ann = pd.read_csv(f"{OUT_DIR}/s5_annotated.csv")
sel = ann[ann["aspect"].isin(["advertisement", "account_support"])].copy()
sel = sel.sample(frac=1, random_state=42).reset_index(drop=True)       # shuffle so aspects are interleaved

blind = pd.DataFrame({
    "item_id": sel["item_id"],
    "aspect": sel["aspect"],
    "aspect_label": sel["aspect"].map(lambda a: ASPECT_TAXONOMY[a]["label"]),
    "aspect_description": sel["aspect"].map(lambda a: ASPECT_TAXONOMY[a]["description"]),
    "sentence_text": sel["sentence_text"],
    "annotator2_label": ""})                                            # negative / neutral / positive / na
key = sel[["item_id", "aspect", "human_label"]].rename(columns={"human_label": "annotator1_label"})

bp = safe_path(f"{OUT_DIR}/s5_verify_blind_for_annotator2.csv")
kp = safe_path(f"{OUT_DIR}/s5_verify_key_annotator1_DO_NOT_SHARE.csv")
blind.to_csv(bp, index=False); key.to_csv(kp, index=False)
print(f"Blind file: {bp} ({len(blind)} items)\nKey file:   {kp}")
print("Items per aspect:", blind["aspect"].value_counts().to_dict())

In [ ]:
# ===== Annotator-2 verification tool (standalone; needs only the blind CSV) =====
import os, html
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

try:                                                    # mount Drive if running in Colab
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

# Folder that contains s5_verify_blind_for_annotator2.csv (change if your Drive path differs)
FOLDER     = "/content/drive/MyDrive/GameSense/outputs/supplementary"
BLIND_PATH = f"{FOLDER}/s5_verify_blind_for_annotator2.csv"
WORK_PATH  = f"{FOLDER}/s5_verify_annotated_by_annotator2.csv"   # progress is auto-saved here after every click
LABEL_COL  = "annotator2_label"

def annotate_verify():
    # Resume from the working file if it exists; otherwise start from the blind file
    if os.path.exists(WORK_PATH):
        df = pd.read_csv(WORK_PATH)
    else:
        df = pd.read_csv(BLIND_PATH)
        df.to_csv(WORK_PATH, index=False)
        print("Created working file:", WORK_PATH)
    df[LABEL_COL] = df[LABEL_COL].fillna("").astype(str).str.strip().str.lower()

    state = {"i": None, "hist": []}
    head, body = widgets.HTML(), widgets.HTML()
    rules = widgets.HTML(
        "<div style='font-size:12px;color:#999;margin-top:8px;line-height:1.6'>"
        "<b>Judge the attitude toward the aspect, not the whole sentence.</b><br>"
        "&bull; <b>na</b>: the aspect is not actually discussed (also pure emotion, feature requests, truncated fragments).<br>"
        "&bull; Negating a problem is <b>positive</b>: e.g. 'no ads', 'ad-free', 'no cheaters'.<br>"
        "&bull; Problem resolved / thanking support is <b>positive</b>: e.g. 'my account was restored', 'support was helpful'."
        "</div>")

    def pending():
        return df.index[df[LABEL_COL] == ""].tolist()

    def show():
        t = pending()
        done = len(df) - len(t)
        if not t:
            head.value = f"<h3>All done ({done}/{len(df)}). Thank you!</h3>"
            body.value, state["i"] = "", None
            return
        i = t[0]
        state["i"] = i
        r = df.loc[i]
        head.value = f"<b>Progress {done}/{len(df)}</b>"
        body.value = (f"<p style='font-size:19px'>{html.escape(str(r['sentence_text']))}</p>"
                      f"<p style='color:#8fd0ff'>Aspect: <b>{html.escape(str(r['aspect_label']))}</b><br>"
                      f"<span style='font-size:12px;color:#aaa'>{html.escape(str(r['aspect_description']))}</span></p>")

    def save():
        df.to_csv(WORK_PATH, index=False)

    def set_label(v):
        if state["i"] is None:
            return
        df.at[state["i"], LABEL_COL] = v
        state["hist"].append(state["i"])
        save()
        show()

    def undo(_):
        if not state["hist"]:
            return
        i = state["hist"].pop()
        df.at[i, LABEL_COL] = ""
        save()
        show()

    buttons = []
    for v, style, text in [("negative", "danger", "negative"),
                           ("neutral", "", "neutral"),
                           ("positive", "success", "positive"),
                           ("na", "warning", "na (aspect not discussed)")]:
        b = widgets.Button(description=text, button_style=style, layout=widgets.Layout(width="200px"))
        b.on_click(lambda _, v=v: set_label(v))
        buttons.append(b)
    u = widgets.Button(description="Undo last", layout=widgets.Layout(width="140px"))
    u.on_click(undo)
    buttons.append(u)

    show()
    display(widgets.VBox([head, body, widgets.HBox(buttons), rules]))
    return df, state, buttons          # returned for testing only; the call below discards it

_ = annotate_verify()

In [ ]:
# ===== S7a: Is there enough date/version coverage for the version analysis? (CPU only; needs S0) =====
MIN_N        = 100     # proposal: >=100 reviews per window
WINDOW_DAYS  = 14      # proposal: preferably 14-day windows
MIN_VER_N    = 100     # a version must have at least this many reviews to be a candidate "update"
MIN_POST_SHARE = 0.40  # share of post-window reviews that are on the new version (else the comparison is diluted)

rows, cands = [], []
for slug in GAME_SLUGS:
    p = f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv"
    if not os.path.exists(p):
        print("missing:", p); continue
    d = pd.read_csv(p, usecols=["reviewId", "at", "reviewCreatedVersion"])
    d["at"] = pd.to_datetime(d["at"], errors="coerce")
    d = d.dropna(subset=["at"])
    d["ver"] = d["reviewCreatedVersion"].astype("string")          # versions may be floats/ints/strings
    lo, hi = d["at"].min(), d["at"].max()
    w = d.groupby(pd.Grouper(key="at", freq=f"{WINDOW_DAYS}D")).size()
    vc = d["ver"].value_counts()
    big = vc[vc >= MIN_VER_N]
    rows.append({"game": slug, "n_reviews": len(d), "from": lo.date(), "to": hi.date(), "span_days": (hi - lo).days,
                 "version_missing_%": round(100 * d["ver"].isna().mean(), 1), "n_versions": int(vc.shape[0]),
                 "versions>=%d" % MIN_VER_N: int(len(big)),
                 "min_full_window_n": int(w.iloc[:-1].min()) if len(w) > 1 else int(w.min())})
    for v in big.index:                                              # candidate update dates
        dv = d.loc[d["ver"] == v, "at"]
        t0 = dv.quantile(0.05)                                       # robust "first seen" date (ignores stragglers)
        pre  = d[(d["at"] >= t0 - pd.Timedelta(days=WINDOW_DAYS)) & (d["at"] < t0)]
        post = d[(d["at"] >= t0) & (d["at"] < t0 + pd.Timedelta(days=WINDOW_DAYS))]
        covered = (t0 - pd.Timedelta(days=WINDOW_DAYS) >= lo) and (t0 + pd.Timedelta(days=WINDOW_DAYS) <= hi)
        share = float((post["ver"] == v).mean()) if len(post) else np.nan
        cands.append({"game": slug, "version": v, "version_n": int(vc[v]), "first_seen(p05)": t0.date(),
                      "n_pre": len(pre), "n_post": len(post), "post_share_on_new": round(share, 2),
                      "windows_in_range": covered,
                      "usable": bool(covered and len(pre) >= MIN_N and len(post) >= MIN_N and share >= MIN_POST_SHARE)})

print("=== Coverage per game ===")
print(pd.DataFrame(rows).to_string(index=False))
c = pd.DataFrame(cands).sort_values(["game", "first_seen(p05)"])
print("\n=== Candidate version transitions (14-day pre/post windows) ===")
print(c.to_string(index=False))
print("\nUsable transitions per game:", c[c["usable"]].groupby("game").size().to_dict())
c.to_csv(safe_path(f"{OUT_DIR}/version_candidates.csv"), index=False)

In [ ]:
# ===== S7b: Check that date/version metadata survived into the prediction files (CPU only; needs S0) =====
import pyarrow.parquet as pq

rows = []
for slug in GAME_SLUGS:
    clean = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv", usecols=["reviewId", "at", "reviewCreatedVersion"])
    clean["at"] = pd.to_datetime(clean["at"], errors="coerce")
    row = {"game": slug, "clean_at_range": f"{clean['at'].min().date()} -> {clean['at'].max().date()}",
           "clean_at_null_%": round(100 * clean["at"].isna().mean(), 1)}
    for kind in ["aspects", "pairs"]:
        p = f"{PROJECT_ROOT}/outputs/predictions/{slug}_{kind}.parquet"
        names = pq.ParquetFile(p).schema.names
        want = [c for c in ["reviewId", "at", "reviewCreatedVersion"] if c in names]
        d = pd.read_parquet(p, columns=want)
        if "at" in d.columns:
            d["at"] = pd.to_datetime(d["at"], errors="coerce")
            ok = d["at"].notna().any()
            row[f"{kind}_at_null_%"] = round(100 * d["at"].isna().mean(), 1)
            row[f"{kind}_at_range"] = f"{d['at'].min().date()} -> {d['at'].max().date()}" if ok else "all missing"
        else:
            row[f"{kind}_at_null_%"] = "column missing"; row[f"{kind}_at_range"] = "-"
        row[f"{kind}_version_null_%"] = (round(100 * d["reviewCreatedVersion"].isna().mean(), 1)
                                         if "reviewCreatedVersion" in d.columns else "column missing")
    rows.append(row)

res = pd.DataFrame(rows).set_index("game").T
print(res.to_string())

In [ ]:
# ===== S7c: Why are 20-31% of review timestamps unparsed? (CPU only; needs S0; read-only) =====
def parse_default(s):
    """What the earlier steps effectively did: format is inferred from the first value, non-matching rows become NaT."""
    return pd.to_datetime(s, errors="coerce")

def parse_robust(s):
    """Parse every value on its own, accept ISO strings with/without fractions or timezone; result is naive UTC."""
    try:
        t = pd.to_datetime(s, format="mixed", utc=True, errors="coerce")
    except (ValueError, TypeError):                                  # older pandas: slower per-value fallback
        from dateutil import parser
        t = pd.to_datetime(s.map(lambda x: parser.parse(x) if isinstance(x, str) and x.strip() else pd.NaT),
                           errors="coerce", utc=True)
    return t.dt.tz_localize(None)

rows, formats, examples = [], {}, {}
for slug in GAME_SLUGS:
    raw = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv", usecols=["reviewId", "at"], dtype={"at": str})
    s = raw["at"]
    empty = s.isna() | (s.astype(str).str.strip() == "")
    d1, d2 = parse_default(s), parse_robust(s)
    rows.append({"game": slug, "n_rows": len(raw),
                 "empty_%": round(100 * empty.mean(), 1),
                 "parsed_default_%": round(100 * d1.notna().mean(), 1),
                 "parsed_robust_%": round(100 * d2.notna().mean(), 1),
                 "range_default": f"{d1.min().date()} -> {d1.max().date()}",
                 "range_robust": f"{d2.min().date()} -> {d2.max().date()}",
                 "span_days_robust": int((d2.max() - d2.min()).days)})
    formats[slug] = s.dropna().astype(str).str.replace(r"\d", "9", regex=True).value_counts().head(4).to_dict()
    examples[slug] = raw.loc[d1.isna() & d2.notna(), "at"].head(4).tolist()

pd.set_option("display.width", 250)
print(pd.DataFrame(rows).set_index("game").T.to_string())
print("\n=== Timestamp formats (digits shown as 9), top 4 per game ===")
for k, v in formats.items():
    print(f"{k:11s}", v)
print("\n=== Example strings that the default parser dropped but the robust parser recovered ===")
for k, v in examples.items():
    print(f"{k:11s}", v)

In [ ]:
# ===== S7a-v2: Version-coverage diagnostic with robust timestamp parsing (CPU only; needs S0; read-only except one csv) =====
MIN_N, WINDOW_DAYS, MIN_VER_N, MIN_POST_SHARE = 100, 14, 100, 0.40
LAST_SCRAPE = pd.Timestamp("2026-08-22")      # no review can be later than the last scrape

def parse_robust(s):
    """Parse every value on its own (ISO and M/D/YYYY H:MM mixed); result is naive UTC."""
    try:
        t = pd.to_datetime(s, format="mixed", utc=True, errors="coerce")
    except (ValueError, TypeError):
        from dateutil import parser
        t = pd.to_datetime(s.map(lambda x: parser.parse(x) if isinstance(x, str) and x.strip() else pd.NaT),
                           errors="coerce", utc=True)
    return t.dt.tz_localize(None)

rows, cands, checks = [], [], []
for slug in GAME_SLUGS:
    d = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv",
                    usecols=["reviewId", "at", "reviewCreatedVersion"], dtype={"at": str})
    # --- sanity check: are slash dates month-first (M/D/YYYY)? first field >12 would indicate day-first ---
    m = d["at"].str.extract(r"^(\d{1,2})/(\d{1,2})/\d{4}").dropna().astype(int)
    d["at"] = parse_robust(d["at"])
    checks.append({"game": slug, "slash_dates": len(m), "first_field>12 (day-first?)": int((m[0] > 12).sum()),
                   "second_field>12 (month-first)": int((m[1] > 12).sum()),
                   "unparsed": int(d["at"].isna().sum()), "dates_after_last_scrape": int((d["at"] > LAST_SCRAPE).sum())})
    d = d.dropna(subset=["at"])
    d["ver"] = d["reviewCreatedVersion"].astype("string").str.split("_").str[0]   # Genshin: 6.7.0_4548..._4576... -> 6.7.0
    lo, hi = d["at"].min(), d["at"].max()
    w = d.groupby(pd.Grouper(key="at", freq=f"{WINDOW_DAYS}D")).size()
    vc = d["ver"].value_counts()
    big = vc[vc >= MIN_VER_N]
    rows.append({"game": slug, "n_reviews": len(d), "from": lo.date(), "to": hi.date(), "span_days": (hi - lo).days,
                 "version_missing_%": round(100 * d["ver"].isna().mean(), 1), "n_versions": int(vc.shape[0]),
                 f"versions>={MIN_VER_N}": int(len(big)),
                 "min_full_window_n": int(w.iloc[:-1].min()) if len(w) > 1 else int(w.min())})
    for v in big.index:
        t0 = d.loc[d["ver"] == v, "at"].quantile(0.05)                      # robust "first seen" date
        pre  = d[(d["at"] >= t0 - pd.Timedelta(days=WINDOW_DAYS)) & (d["at"] < t0)]
        post = d[(d["at"] >= t0) & (d["at"] < t0 + pd.Timedelta(days=WINDOW_DAYS))]
        covered = (t0 - pd.Timedelta(days=WINDOW_DAYS) >= lo) and (t0 + pd.Timedelta(days=WINDOW_DAYS) <= hi)
        share = float((post["ver"] == v).mean()) if len(post) else np.nan
        cands.append({"game": slug, "version": v, "version_n": int(vc[v]), "first_seen(p05)": t0.date(),
                      "n_pre": len(pre), "n_post": len(post), "post_share_on_new": round(share, 2),
                      "windows_in_range": covered,
                      "usable": bool(covered and len(pre) >= MIN_N and len(post) >= MIN_N and share >= MIN_POST_SHARE)})

pd.set_option("display.width", 250)
print("=== Date sanity checks ===");  print(pd.DataFrame(checks).to_string(index=False))
print("\n=== Coverage per game (robust parsing) ===");  print(pd.DataFrame(rows).to_string(index=False))
c = pd.DataFrame(cands).sort_values(["game", "first_seen(p05)"])
print("\n=== Candidate version transitions (usable ones only) ===")
print(c[c["usable"]].to_string(index=False))
print("\nUsable transitions per game:", c[c["usable"]].groupby("game").size().to_dict())
c.to_csv(safe_path(f"{OUT_DIR}/version_candidates_v2.csv"), index=False)

In [ ]:
# ===== S7d: Version-aware pre/post analysis (CPU only; needs S0 and S7a-v2's version_candidates_v2.csv) =====
# Descriptive only: windows before/after a version's first-seen date. No causal claim.
TRANSITIONS = [("royalmatch", "37281.0"), ("royalmatch", "37743.0"), ("genshin", "6.7.0")]   # (game, version) from S7a-v2
WINDOW_DAYS, N_BOOT, MIN_MENT = 14, 2000, 20
CAUTION = {"advertisement": "sentiment failed S5 check", "account_support": "sentiment failed S5 check",
           "cheating": "sentiment borderline", "paywall": "sentiment not audited"}
COMMON = ["performance_stability", "monetization_value", "advertisement", "level_difficulty_balance",
          "content_progression", "paywall", "account_support"]
GENRE = {"royalmatch": [], "genshin": ["gacha_rates", "character_operator_balance"],
         "mlbb": ["character_operator_balance", "matchmaking", "cheating"], "pubgm": ["cheating", "matchmaking"],
         "arknights": ["gacha_rates", "character_operator_balance", "grind_progression"]}

def parse_robust(s):
    try:
        t = pd.to_datetime(s, format="mixed", utc=True, errors="coerce")
    except (ValueError, TypeError):
        from dateutil import parser
        t = pd.to_datetime(s.map(lambda x: parser.parse(x) if isinstance(x, str) and x.strip() else pd.NaT),
                           errors="coerce", utc=True)
    return t.dt.tz_localize(None)

cand = pd.read_csv(f"{OUT_DIR}/version_candidates_v2.csv")
rng = np.random.default_rng(42)

def boot_diff(pre, post, fn):
    """Cluster bootstrap over reviews (multinomial weights). fn(arrays, weights)->(A,) statistic. Returns point, lo, hi of post-pre."""
    def stat(arrs, W):
        return np.stack([fn(arrs, W[b]) for b in range(W.shape[0])])
    Wp = rng.multinomial(len(pre[0]), np.ones(len(pre[0])) / len(pre[0]), size=N_BOOT)
    Wq = rng.multinomial(len(post[0]), np.ones(len(post[0])) / len(post[0]), size=N_BOOT)
    d = stat(post, Wq) - stat(pre, Wp)
    pt = fn(post, np.ones(len(post[0]))) - fn(pre, np.ones(len(pre[0])))
    return pt, np.percentile(d, 2.5, axis=0), np.percentile(d, 97.5, axis=0)

all_rows = []
for slug, ver in TRANSITIONS:
    c = cand[(cand["game"] == slug) & (cand["version"].astype(str) == ver)]
    assert len(c) == 1, f"transition not found in version_candidates_v2.csv: {slug} {ver}"
    t0 = pd.Timestamp(c["first_seen(p05)"].iloc[0])
    aspects = [a for a in ASPECT_IDS if a in (COMMON + GENRE[slug]) and a != "grind_progression"]

    rev = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv",
                      usecols=["reviewId", "at", "score", "reviewCreatedVersion"], dtype={"at": str}).drop_duplicates("reviewId")
    rev["at"] = parse_robust(rev["at"])
    rev["ver"] = rev["reviewCreatedVersion"].astype("string").str.split("_").str[0]
    rev = rev.dropna(subset=["at"]).set_index("reviewId")
    sents = pd.read_parquet(f"{PROJECT_ROOT}/outputs/predictions/{slug}_aspects.parquet", columns=["reviewId"])
    pairs = pd.read_parquet(f"{PROJECT_ROOT}/outputs/predictions/{slug}_pairs.parquet", columns=["reviewId", "aspect", "sentiment"])
    ov = pd.read_parquet(f"{PROJECT_ROOT}/outputs/dashboard/overall_{slug}.parquet").drop_duplicates("reviewId").set_index("reviewId")

    S = sents.groupby("reviewId").size().reindex(rev.index).fillna(0).values
    M = pairs.groupby(["reviewId", "aspect"]).size().unstack(fill_value=0).reindex(index=rev.index, columns=aspects).fillna(0).values
    Nn = (pairs[pairs["sentiment"] == "negative"].groupby(["reviewId", "aspect"]).size()
          .unstack(fill_value=0).reindex(index=rev.index, columns=aspects).fillna(0).values)
    stars = pd.to_numeric(rev["score"], errors="coerce").values
    oneg = (ov["overall"].reindex(rev.index) == "negative").astype(float).values

    lo_t, hi_t = t0 - pd.Timedelta(days=WINDOW_DAYS), t0 + pd.Timedelta(days=WINDOW_DAYS)
    pre_m = ((rev["at"] >= lo_t) & (rev["at"] < t0)).values
    post_m = ((rev["at"] >= t0) & (rev["at"] < hi_t)).values
    share_new = float((rev.loc[post_m, "ver"] == ver).mean())
    print(f"\n==================== {slug} | version {ver} | first seen {t0.date()} ====================")
    print(f"pre  [{lo_t.date()} .. {t0.date()}): {pre_m.sum()} reviews, {int(S[pre_m].sum())} sentences")
    print(f"post [{t0.date()} .. {hi_t.date()}): {post_m.sum()} reviews, {int(S[post_m].sum())} sentences | "
          f"share of post reviews on new version (missing version counted as not new): {share_new:.2f}")

    # arrays per window: (S, M, N, stars, overall_neg); k=1 mentions, k=2 negative mentions per 100 sentences
    pre = (S[pre_m], M[pre_m], Nn[pre_m], stars[pre_m], oneg[pre_m])
    post = (S[post_m], M[post_m], Nn[post_m], stars[post_m], oneg[post_m])
    mean = lambda a, W, k: (W @ np.nan_to_num(a[k])) / (W @ (~np.isnan(a[k])).astype(float))

    # ---- overall (review level) ----
    ov_rows = []
    for name, k in [("avg_stars", 3), ("overall_pct_negative (Task1)", 4)]:
        fn = lambda arrs, W, k=k: np.array([mean(arrs, W, k)])
        pt, lo, hi = boot_diff(pre, post, fn)
        scale = 100 if k == 4 else 1
        ov_rows.append({"metric": name, "pre": round(scale * mean(pre, np.ones(len(pre[0])), k), 2),
                        "post": round(scale * mean(post, np.ones(len(post[0])), k), 2),
                        "delta": round(scale * pt[0], 2), "ci95": [round(scale * lo[0], 2), round(scale * hi[0], 2)]})
    print(pd.DataFrame(ov_rows).to_string(index=False))

    # ---- aspect level ----
    res = {}
    for key, k in [("mentions", 1), ("negative", 2)]:
        fn = lambda arrs, W, k=k: 100 * (W @ arrs[k]) / (W @ arrs[0])
        pt, lo, hi = boot_diff(pre, post, fn)
        res[key] = (100 * pre[k].sum(0) / pre[0].sum(), 100 * post[k].sum(0) / post[0].sum(), pt, lo, hi)
    tab = []
    for j, a in enumerate(aspects):
        n_m = int(pre[1][:, j].sum() + post[1][:, j].sum())
        if n_m < MIN_MENT:
            continue
        row = {"game": slug, "version": ver, "aspect": a, "n_mentions_total": n_m}
        for key in ["mentions", "negative"]:
            p0, p1, d, lo, hi = res[key]
            row.update({f"{key}_pre": round(p0[j], 2), f"{key}_post": round(p1[j], 2), f"{key}_delta": round(d[j], 2),
                        f"{key}_ci95": [round(lo[j], 2), round(hi[j], 2)], f"{key}_sig": bool(lo[j] > 0 or hi[j] < 0)})
        row["caution"] = CAUTION.get(a, "")
        tab.append(row)
    t = pd.DataFrame(tab)
    show = t[["aspect", "n_mentions_total", "mentions_pre", "mentions_post", "mentions_delta", "mentions_ci95", "mentions_sig",
              "negative_pre", "negative_post", "negative_delta", "negative_ci95", "negative_sig", "caution"]]
    print("\n(rates are per 100 sentences; delta = post - pre; sig = 95% bootstrap CI excludes 0)")
    print(show.to_string(index=False))
    all_rows.append(t.assign(t0=t0.date(), n_pre_reviews=int(pre_m.sum()), n_post_reviews=int(post_m.sum()), post_share_new=round(share_new, 2)))

pd.concat(all_rows).to_csv(safe_path(f"{OUT_DIR}/version_analysis_results.csv"), index=False)
print("\nSaved -> version_analysis_results.csv")

In [ ]:
# ===== S7e: Aspect mention rate by 14-day window and star band (CPU only; needs S0) =====
# Purpose: separate "more complaints" from "more talk" without relying on the (unvalidated) aspect sentiment.
# Star bands are only used to STRATIFY the descriptive view; they are never used as aspect labels.
GAME, ASPECT, WINDOW_DAYS, N_BOOT = "royalmatch", "advertisement", 14, 1000

def parse_robust(s):
    try:
        t = pd.to_datetime(s, format="mixed", utc=True, errors="coerce")
    except (ValueError, TypeError):
        from dateutil import parser
        t = pd.to_datetime(s.map(lambda x: parser.parse(x) if isinstance(x, str) and x.strip() else pd.NaT),
                           errors="coerce", utc=True)
    return t.dt.tz_localize(None)

rev = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{GAME}_clean.csv", usecols=["reviewId", "at", "score"],
                  dtype={"at": str}).drop_duplicates("reviewId")
rev["at"] = parse_robust(rev["at"]); rev = rev.dropna(subset=["at"]).set_index("reviewId")
rev["band"] = pd.cut(pd.to_numeric(rev["score"], errors="coerce"), [0, 2, 3, 5], labels=["1-2 stars", "3 stars", "4-5 stars"])
sents = pd.read_parquet(f"{PROJECT_ROOT}/outputs/predictions/{GAME}_aspects.parquet", columns=["reviewId"])
pairs = pd.read_parquet(f"{PROJECT_ROOT}/outputs/predictions/{GAME}_pairs.parquet", columns=["reviewId", "aspect"])
rev["S"] = sents.groupby("reviewId").size().reindex(rev.index).fillna(0).values
rev["M"] = pairs[pairs["aspect"] == ASPECT].groupby("reviewId").size().reindex(rev.index).fillna(0).values
t_start = rev["at"].min().normalize()
rev["win"] = ((rev["at"] - t_start).dt.days // WINDOW_DAYS).astype(int)

rng = np.random.default_rng(42)
def rate_ci(d):
    S, M = d["S"].values, d["M"].values
    if S.sum() == 0 or len(d) < 30:
        return np.nan, np.nan, np.nan
    W = rng.multinomial(len(d), np.ones(len(d)) / len(d), size=N_BOOT)
    b = 100 * (W @ M) / np.maximum(W @ S, 1)
    return 100 * M.sum() / S.sum(), np.percentile(b, 2.5), np.percentile(b, 97.5)

rows = []
for w, dw in rev.groupby("win"):
    start = t_start + pd.Timedelta(days=int(w) * WINDOW_DAYS)
    row = {"window_start": start.date(), "n_reviews": len(dw), "n_sentences": int(dw["S"].sum())}
    for label, d in [("all", dw)] + [(b, dw[dw["band"] == b]) for b in ["1-2 stars", "3 stars", "4-5 stars"]]:
        p, lo, hi = rate_ci(d)
        row[f"{label}"] = f"{p:.1f} [{lo:.1f}, {hi:.1f}]" if not np.isnan(p) else "n<30"
        row[f"n_{label}"] = len(d)
    # share of this window's mentions that come from low-star reviews
    row["ad_mentions"] = int(dw["M"].sum())
    row["mentions_from_1-2_stars_%"] = round(100 * dw.loc[dw["band"] == "1-2 stars", "M"].sum() / max(dw["M"].sum(), 1), 1)
    rows.append(row)

out = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(f"{GAME} / {ASPECT}: mentions per 100 sentences [95% bootstrap CI] by {WINDOW_DAYS}-day window and star band")
print(out[["window_start", "n_reviews", "all", "1-2 stars", "3 stars", "4-5 stars", "ad_mentions", "mentions_from_1-2_stars_%"]].to_string(index=False))
print("\nreviews per band:")
print(out[["window_start", "n_1-2 stars", "n_3 stars", "n_4-5 stars"]].to_string(index=False))
out.to_csv(safe_path(f"{OUT_DIR}/mention_rate_by_window_band_{GAME}_{ASPECT}.csv"), index=False)

In [ ]:
# ===== S8: Dashboard data package v2 (CPU only; needs S0). Fixes dates/versions, adds star-band rates, safer examples =====
# Reads predictions and cleaned reviews; writes ONLY to outputs/dashboard_v2/ (the old outputs/dashboard/ is left untouched).
import shutil, glob
OUT_DASH = f"{PROJECT_ROOT}/outputs/dashboard_v2"
os.makedirs(OUT_DASH, exist_ok=True)
WINDOW = {"arknights": 91}                       # days per trend window (sparse game); all other games use 14
CAUTION_ASPECTS = {"advertisement", "account_support"}   # sentiment failed the S5 check -> examples only from 1-2 star reviews
COMMON = ["performance_stability", "monetization_value", "advertisement", "level_difficulty_balance",
          "content_progression", "paywall", "account_support"]
GENRE = {"royalmatch": [], "genshin": ["gacha_rates", "character_operator_balance"],
         "mlbb": ["character_operator_balance", "matchmaking", "cheating"], "pubgm": ["cheating", "matchmaking"],
         "arknights": ["gacha_rates", "character_operator_balance", "grind_progression"]}

def parse_robust(s):
    try:
        t = pd.to_datetime(s, format="mixed", utc=True, errors="coerce")
    except (ValueError, TypeError):
        from dateutil import parser
        t = pd.to_datetime(s.map(lambda x: parser.parse(x) if isinstance(x, str) and x.strip() else pd.NaT),
                           errors="coerce", utc=True)
    return t.dt.tz_localize(None)

metas, pairs_all, trends, examples = [], [], [], []
for slug in GAME_SLUGS:
    clean = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv", dtype={"at": str},
                        usecols=["reviewId", "at", "score", "reviewCreatedVersion", "content_clean", "thumbsUpCount"]).drop_duplicates("reviewId")
    clean["at"] = parse_robust(clean["at"])
    clean["version"] = clean["reviewCreatedVersion"].astype("string").str.split("_").str[0]
    ov = pd.read_parquet(f"{PROJECT_ROOT}/outputs/dashboard/overall_{slug}.parquet").drop_duplicates("reviewId")[["reviewId", "overall"]]
    sents = pd.read_parquet(f"{PROJECT_ROOT}/outputs/predictions/{slug}_aspects.parquet", columns=["reviewId"])
    meta = clean.merge(ov, on="reviewId", how="left")
    meta["game"] = slug
    meta["n_sentences"] = meta["reviewId"].map(sents.groupby("reviewId").size()).fillna(0).astype(int)
    meta["thumbs"] = pd.to_numeric(meta["thumbsUpCount"], errors="coerce").fillna(0)

    pairs = pd.read_parquet(f"{PROJECT_ROOT}/outputs/predictions/{slug}_pairs.parquet",
                            columns=["sentence_uid", "reviewId", "game", "sentence_text", "aspect", "p_aspect", "sentiment", "p_negative"])
    pairs = pairs.merge(meta[["reviewId", "at", "version", "score", "thumbs", "content_clean"]], on="reviewId", how="left")
    pairs = pairs.rename(columns={"score": "stars"})

    # ---- trend table: fixed windows, rates per 100 sentences, overall and by star band ----
    wd = WINDOW.get(slug, 14)
    t0 = meta["at"].min().normalize()
    meta["win"] = ((meta["at"] - t0).dt.days // wd).astype(int)
    meta["band"] = pd.cut(pd.to_numeric(meta["score"], errors="coerce"), [0, 2, 3, 5], labels=["low", "mid", "high"])
    pw = pairs.merge(meta[["reviewId", "win", "band"]], on="reviewId", how="left")
    asp = [a for a in ASPECT_IDS if a in (COMMON + GENRE[slug]) and a != "grind_progression"]
    wins = np.arange(int(meta["win"].max()) + 1)
    tot = meta.groupby("win").agg(n_reviews=("reviewId", "size"), n_sent=("n_sentences", "sum")).reindex(wins, fill_value=0)
    totb = (meta.groupby(["win", "band"], observed=True)["n_sentences"].sum().unstack(fill_value=0)
            .reindex(index=wins, columns=["low", "mid", "high"], fill_value=0))
    def cnt(df):
        return df.groupby(["win", "aspect"]).size().unstack(fill_value=0).reindex(index=wins, columns=asp, fill_value=0)
    men, neg = cnt(pw), cnt(pw[pw["sentiment"] == "negative"])
    men_low, men_high = cnt(pw[pw["band"] == "low"]), cnt(pw[pw["band"] == "high"])
    for a in asp:
        trends.append(pd.DataFrame({
            "game": slug, "aspect": a, "window_start": [t0 + pd.Timedelta(days=int(w) * wd) for w in wins],
            "window_days": wd, "n_reviews": tot["n_reviews"].values, "n_sentences": tot["n_sent"].values,
            "mentions": men[a].values, "negative_mentions": neg[a].values,
            "mentions_per100": 100 * men[a].values / np.where(tot["n_sent"].values > 0, tot["n_sent"].values, np.nan),
            "negative_per100": 100 * neg[a].values / np.where(tot["n_sent"].values > 0, tot["n_sent"].values, np.nan),
            "mentions_per100_1to2star": 100 * men_low[a].values / np.where(totb["low"].values > 0, totb["low"].values, np.nan),
            "mentions_per100_4to5star": 100 * men_high[a].values / np.where(totb["high"].values > 0, totb["high"].values, np.nan),
            "reliable_window": tot["n_reviews"].values >= 100}))

    # ---- representative negative examples; failed-sentiment aspects only from 1-2 star reviews ----
    ex = pairs[pairs["sentiment"] == "negative"].copy()
    ex = ex[~(ex["aspect"].isin(CAUTION_ASPECTS) & ~(ex["stars"] <= 2))]
    ex["rank_score"] = ex["p_aspect"] * ex["p_negative"]
    ex = (ex.sort_values(["aspect", "rank_score", "thumbs"], ascending=[True, False, False]).groupby("aspect").head(30))
    examples.append(ex[["game", "aspect", "sentence_text", "content_clean", "stars", "p_aspect", "p_negative",
                        "rank_score", "thumbs", "version", "at"]])
    metas.append(meta[["game", "reviewId", "at", "version", "score", "overall", "n_sentences", "thumbs"]])
    pairs_all.append(pairs[["game", "sentence_uid", "reviewId", "aspect", "p_aspect", "sentiment", "p_negative",
                            "stars", "at", "version", "sentence_text"]])

meta_all, pairs_df = pd.concat(metas, ignore_index=True), pd.concat(pairs_all, ignore_index=True)
trend_df, ex_df = pd.concat(trends, ignore_index=True), pd.concat(examples, ignore_index=True)
for name, df in [("reviews_meta.parquet", meta_all), ("pairs_v2.parquet", pairs_df), ("examples_v2.parquet", ex_df)]:
    df.to_parquet(safe_path(f"{OUT_DASH}/{name}"), index=False)
trend_df.to_csv(safe_path(f"{OUT_DASH}/trend_v2.csv"), index=False)

# ---- copy tables the dashboard also needs ----
for src in [f"{PROJECT_ROOT}/outputs/dashboard/profile.csv", f"{OUT_DIR}/version_analysis_results.csv",
            f"{OUT_DIR}/version_candidates_v2.csv"] + glob.glob(f"{OUT_DIR}/mention_rate_by_window_band_*.csv"):
    if os.path.exists(src):
        shutil.copy(src, safe_path(f"{OUT_DASH}/{os.path.basename(src)}"))

# ---- checks ----
print("Files in", OUT_DASH, ":", sorted(os.listdir(OUT_DASH)))
chk = meta_all.groupby("game").agg(reviews=("reviewId", "size"), first=("at", "min"), last=("at", "max"),
                                   at_null=("at", lambda s: round(100 * s.isna().mean(), 1)),
                                   version_null=("version", lambda s: round(100 * s.isna().mean(), 1)),
                                   overall_null=("overall", lambda s: round(100 * s.isna().mean(), 1)))
print(chk.to_string())
print("pairs:", len(pairs_df), "| pairs with missing date:", int(pairs_df["at"].isna().sum()),
      "| trend rows:", len(trend_df), "| example rows:", len(ex_df))

In [ ]:
# ===== S9a: Task 1 audit - split reconstruction, duplicate-text leakage, predicted-vs-weak label shares (CPU only; needs S0) =====
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

EXPECT_SIZES = (23783, 5097, 5097)        # train/val/test sizes printed by the original Task 1 notebook
EXPECT_BERT_F1 = 0.664                    # test macro-F1 reported by the original notebook
NEAR_DUP, NEAR_THR = True, 0.90
NEGATORS = {"no", "not", "never", "nor", "nothing", "cannot", "without", "none", "nobody", "neither"}
STOP = list(ENGLISH_STOP_WORDS - NEGATORS)       # keep negators, otherwise "not good" and "good" look identical
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# ---------- 1. Rebuild the Task 1 split exactly as in the training notebook ----------
frames = []
for slug in GAME_SLUGS:
    d = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv")
    d["game"] = slug
    frames.append(d)
reviews_df = pd.concat(frames, ignore_index=True).dropna(subset=["content_clean", "overall_sentiment_weak"])
reviews_df["label_id"] = reviews_df["overall_sentiment_weak"].map(LABEL2ID)
train_r, temp_r = train_test_split(reviews_df, test_size=0.30, random_state=42, stratify=reviews_df["label_id"])
val_r, test_r = train_test_split(temp_r, test_size=0.50, random_state=42, stratify=temp_r["label_id"])
sizes = (len(train_r), len(val_r), len(test_r))
print("split sizes (train/val/test):", sizes, "| expected:", EXPECT_SIZES, "->", "match" if sizes == EXPECT_SIZES else "MISMATCH")

# ---------- 2. BERT predictions for the test rows (already saved in overall_*.parquet; no inference needed) ----------
ov = (pd.concat([pd.read_parquet(f"{PROJECT_ROOT}/outputs/dashboard/overall_{s}.parquet") for s in GAME_SLUGS])
      .drop_duplicates("reviewId")[["reviewId", "overall"]])
test_r = test_r.merge(ov, on="reviewId", how="left")
assert test_r["overall"].notna().all(), "some test reviews have no saved BERT prediction"
test_r["bert_pred"] = test_r["overall"].map(LABEL2ID)
f1_bert_full = f1_score(test_r["label_id"], test_r["bert_pred"], average="macro")
print(f"BERT test macro-F1 recomputed from saved predictions: {f1_bert_full:.3f} (original notebook: {EXPECT_BERT_F1})")
if abs(f1_bert_full - EXPECT_BERT_F1) > 0.005:
    print("WARNING: does not reproduce the original number; the reconstructed split may differ. Interpret the leakage numbers with caution.")

# ---------- 3. Duplicate-text leakage between test and train ----------
norm = lambda t: re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(t).lower())).strip()
for d in (train_r, val_r, test_r):
    d["norm"] = d["content_clean"].map(norm)
train_set = set(train_r["norm"])
test_r["leak_exact"] = test_r["norm"].isin(train_set)
test_r["leak_near"] = False
if NEAR_DUP:
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.2, stop_words=STOP)
    vec.fit(pd.concat([train_r["norm"], test_r["norm"]]))
    Xtr, Xte = vec.transform(train_r["norm"]), vec.transform(test_r["norm"])
    best = np.zeros(len(test_r))
    for s in range(0, Xte.shape[0], 500):                       # chunked cosine similarity (vectors are L2-normalised)
        sim = Xte[s:s + 500] @ Xtr.T
        best[s:s + 500] = sim.max(axis=1).toarray().ravel()
    test_r["leak_near"] = best >= NEAR_THR
test_r["leak_any"] = test_r["leak_exact"] | test_r["leak_near"]
leak = pd.DataFrame({"n_test": len(test_r), "exact_dup_in_train": int(test_r["leak_exact"].sum()),
                     "near_dup_in_train (cos>=%.2f)" % NEAR_THR: int(test_r["leak_near"].sum()),
                     "any": int(test_r["leak_any"].sum()), "any_%": round(100 * test_r["leak_any"].mean(), 1)}, index=["test"])
print("\n=== Test reviews whose text also appears in train ===")
print(leak.T.to_string(header=False))
print("share leaked by weak label (%):", (100 * test_r.groupby("overall_sentiment_weak")["leak_any"].mean()).round(1).to_dict())
for c in ["is_long_outlier", "same_user_repeated_content"]:
    print(f"flag column {c}:", int(reviews_df[c].sum()) if c in reviews_df.columns else "not present in the current clean files")
_cols = [set(f.columns) for f in frames]
_common = set.intersection(*_cols)
print("columns that differ between games:", {slug: sorted(c - _common) for slug, c in zip(GAME_SLUGS, _cols) if c - _common} or "none")
print("examples of leaked texts:", test_r.loc[test_r["leak_any"], "norm"].value_counts().head(6).to_dict())

# ---------- 4. Sensitivity: macro-F1 on all test rows vs. test rows without exact / near duplicates of train ----------
vec1 = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50000, sublinear_tf=True)    # same config as the original baseline
clf1 = LogisticRegression(max_iter=2000, class_weight="balanced").fit(vec1.fit_transform(train_r["content_clean"]), train_r["label_id"])
test_r["tfidf_pred"] = clf1.predict(vec1.transform(test_r["content_clean"]))
subsets = [("all test rows", np.ones(len(test_r), bool)), ("no exact duplicate of train", ~test_r["leak_exact"].values),
           ("no exact/near duplicate of train", ~test_r["leak_any"].values)]
rows = []
for name, col in [("TF-IDF + LR", "tfidf_pred"), ("BERT", "bert_pred")]:
    for sname, mask in subsets:
        d = test_r[mask]
        f = f1_score(d["label_id"], d[col], average=None, labels=[0, 1, 2], zero_division=0)
        rows.append({"model": name, "subset": sname, "n": len(d), "macro_F1": round(float(f.mean()), 3),
                     "neg_F1": round(float(f[0]), 3), "neutral_F1": round(float(f[1]), 3), "pos_F1": round(float(f[2]), 3)})
sens = pd.DataFrame(rows)
print("\n=== Sensitivity to leaked duplicates ===")
print(sens.to_string(index=False))

# ---------- 5. Predicted vs weak-label shares per game (all reviews) ----------
allr = reviews_df.merge(ov, on="reviewId", how="left")
share = lambda s, v: round(100 * (s == v).mean(), 1)
tab = []
for g, d in allr.groupby("game"):
    tab.append({"game": g, "n": len(d),
                "weak_neg": share(d["overall_sentiment_weak"], "negative"), "pred_neg": share(d["overall"], "negative"),
                "weak_neu": share(d["overall_sentiment_weak"], "neutral"), "pred_neu": share(d["overall"], "neutral"),
                "weak_pos": share(d["overall_sentiment_weak"], "positive"), "pred_pos": share(d["overall"], "positive")})
print("\n=== Share of each class (%), star-derived weak label vs BERT prediction ===")
print(pd.DataFrame(tab).to_string(index=False))
print("\nTest-set confusion (rows = weak label, cols = BERT; neg/neu/pos):")
print(pd.crosstab(test_r["label_id"], test_r["bert_pred"]).reindex(index=[0, 1, 2], columns=[0, 1, 2], fill_value=0).values)

update_summary("task1_audit", {
    "split_sizes": list(sizes), "split_matches_original": sizes == EXPECT_SIZES,
    "bert_test_macro_f1_recomputed": round(f1_bert_full, 3),
    "test_rows_with_train_duplicate": {"exact": int(test_r["leak_exact"].sum()), "near": int(test_r["leak_near"].sum()),
                                       "any": int(test_r["leak_any"].sum()), "n_test": len(test_r)},
    "sensitivity": sens.to_dict(orient="records"),
    "class_shares_by_game": tab})
print("\nS9a done -> bert_summary_v2.json['task1_audit']")

In [ ]:
# ===== S9c: Gold label consistency audit on the edge-case patterns found in S5 (CPU only; needs S0; read-only except one csv) =====
import re
CUES = {   # negated-problem / positive-experience cues per aspect (same idea as the S5a cue lists)
 "performance_stability": r"\b(?:smooth|smoothly|no lag|lag[- ]?free|without lag|runs? (?:well|great|fine|perfectly)|stable|optimi[sz]ed|no crash(?:es)?|never crash(?:es)?|no bugs?|bug[- ]?free|no issues?|works? (?:well|great|perfectly|fine)|loads? fast)\b",
 "matchmaking": r"\b(?:fair|balanced|well[- ]balanced|even match(?:es)?|(?:good|great) match(?:es|making)?|match ?making (?:is|was|has been) (?:good|great|fair|fine|better|improved))\b",
 "cheating": r"\b(?:no (?:cheaters?|hackers?|cheating|hacks?)|cheat[- ]?free|hacker[- ]?free|(?:good|great) anti[- ]?cheat|anti[- ]?cheat (?:is|works|has)|fair play|without (?:any )?cheat(?:ing|ers?)?)\b",
 "account_support": r"\b(?:great|good|excellent|amazing|fast|quick|helpful|friendly|responsive) (?:customer )?(?:support|service|response|team|staff)\b|\b(?:support|customer service|team|developers?) (?:is|are|was|were) (?:great|good|helpful|responsive|quick|fast)\b|\b(?:resolved|recovered|restored|fixed) (?:my|the) (?:account|issue|problem)\b|\beasy to (?:recover|restore)\b",
 "advertisement": r"\b(?:no ads?|ad[- ]?free|few ads?|not (?:too )?many ads?|(?:ads?|adverts?) (?:are|is) (?:fine|ok|okay|optional|not (?:bad|annoying|intrusive|too bad))|optional ads?|rewarded ads?|not intrusive|without ads?|doesn.?t (?:have|show) (?:any )?ads?)\b",
}
RESOLVED = r"\b(?:restored|recovered|fixed|resolved|solved|sorted|thank you|thanks)\b"
SUPPORT_WORDS = r"\b(?:account|support|customer|service|team|developers?|staff|server)\b"
SUGGEST = r"\b(?:should|shouldn.?t|please (?:add|make|fix|bring|give|let|allow|remove)|would be (?:nice|great|good|better|cool)|i wish|hope (?:they|you|that)|suggest|needs? to|add (?:a|an|more)|can you|could you|why (?:don.?t|not)|i.?d like|i would like)\b"

g = pd.read_csv(GOLD_PATH)
for a in ASPECT_IDS:
    g[f"gold_{a}"] = pd.to_numeric(g[f"gold_{a}"], errors="coerce").fillna(0).astype(int)
txt = g["sentence_text"].astype(str)
has = lambda pat: txt.str.contains(pat, flags=re.I, regex=True, na=False)
rows = []
def add(pattern, aspect, r, status):
    rows.append({"pattern": pattern, "aspect": aspect, "sentence_uid": r["sentence_uid"], "game": r["game"],
                 "gold_labels": "; ".join(f"{a}:{r.get(f'gold_{a}_sentiment', '') or '?'}" for a in ASPECT_IDS if r[f"gold_{a}"] == 1) or "(none)",
                 "status": status, "sentence_text": r["sentence_text"]})

# A. negated-problem / positive-experience cues: if the aspect is labeled, the sentiment should be positive
for a, pat in CUES.items():
    for _, r in g[has(pat)].iterrows():
        if r[f"gold_{a}"] == 1:
            s = str(r.get(f"gold_{a}_sentiment", "")).strip()
            add("A: negated/positive cue", a, r, "OK" if s == "positive" else f"CHECK: cue present, aspect labeled {s or '?'} (expected positive)")
        else:
            add("A: negated/positive cue", a, r, "CHECK: cue present, aspect not labeled")

# B. problem resolved / gratitude near account or support words -> account_support, expected positive
for _, r in g[has(RESOLVED) & has(SUPPORT_WORDS)].iterrows():
    if r["gold_account_support"] == 1:
        s = str(r.get("gold_account_support_sentiment", "")).strip()
        add("B: resolved/thanks", "account_support", r, "OK" if s == "positive" else f"CHECK: labeled {s or '?'} (expected positive)")
    else:
        add("B: resolved/thanks", "account_support", r, "CHECK: not labeled account_support (may be fine if about performance)")

# C. suggestion-like sentences: rule says feature suggestions get no aspect
for _, r in g[has(SUGGEST)].iterrows():
    n = sum(r[f"gold_{a}"] for a in ASPECT_IDS)
    add("C: suggestion-like", "-", r, "CHECK: has aspect labels (rule: suggestions -> none; complaints phrased as 'should fix' may be valid)" if n else "OK")

res = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 110)
print(f"gold sentences: {len(g)}")
print(res.assign(ok=res["status"].eq("OK")).groupby("pattern")["ok"].agg(matches="size", ok="sum").assign(to_check=lambda d: d["matches"] - d["ok"]).to_string())
chk = res[res["status"] != "OK"].sort_values(["pattern", "aspect"])
print(f"\n=== {len(chk)} rows to check (also saved to gold_consistency_review.csv) ===")
print(chk[["pattern", "aspect", "gold_labels", "status", "sentence_text"]].head(60).to_string(index=False))
res.to_csv(safe_path(f"{OUT_DIR}/gold_consistency_review.csv"), index=False)

In [ ]:
# ===== S9d: Evidence table per aspect (CPU only; needs S0; read-only except one csv + summary json) =====
from sklearn.metrics import roc_auc_score
THR = 0.30                                      # deployed Task 2 threshold
COMMON = ["performance_stability", "monetization_value", "advertisement", "level_difficulty_balance",
          "content_progression", "paywall", "account_support"]
GENRE = {"royalmatch": [], "genshin": ["gacha_rates", "character_operator_balance"],
         "mlbb": ["character_operator_balance", "matchmaking", "cheating"], "pubgm": ["cheating", "matchmaking"],
         "arknights": ["gacha_rates", "character_operator_balance", "grind_progression"]}

# ---------- Task 2: per-aspect precision / recall from the saved out-of-fold probabilities ----------
g = pd.read_csv(GOLD_PATH)
Y = np.stack([pd.to_numeric(g[f"gold_{a}"], errors="coerce").fillna(0).astype(int).values for a in ASPECT_IDS], axis=1)
allowed = np.array([[a in set(COMMON) | set(GENRE[gm]) for a in ASPECT_IDS] for gm in g["game"]])
oof2 = np.load(f"{PROJECT_ROOT}/outputs/oof_plain.npy")
assert oof2.shape == Y.shape
P2 = (oof2 * allowed) >= THR                    # same applicability mask as in deployment

# ---------- Task 3: per-aspect sentiment evidence from the saved grouped-CV predictions ----------
rows3 = []
for _, r in g.iterrows():
    for a in ASPECT_IDS:
        if r[f"gold_{a}"] == 1 and isinstance(r.get(f"gold_{a}_sentiment"), str) and r[f"gold_{a}_sentiment"] in L2I:
            rows3.append({"aspect": a, "label": r[f"gold_{a}_sentiment"], "source": "gold"})
aud = pd.read_csv(AUDIT_PATH)
aud["human_label"] = aud["human_label"].fillna("").astype(str)
aud = aud[aud["human_label"].isin(SENT_LABELS)]
rows3 += [{"aspect": r["aspect"], "label": r["human_label"], "source": "audit"} for _, r in aud.iterrows()]
d3 = pd.DataFrame(rows3)
d3["y"] = d3["label"].map(L2I)
oof3 = np.load(f"{PROJECT_ROOT}/outputs/task3_oof_group.npy")
assert oof3.shape[0] == len(d3), "row order/length mismatch with the saved Task 3 OOF (re-run S2 checks)"
d3["pred"] = oof3.argmax(1); d3["p_neg"] = oof3[:, 0]

# ---------- S5 verdicts (if S5b was run) ----------
s5 = {}
if os.path.exists(SUMMARY_PATH):
    s5 = json.load(open(SUMMARY_PATH)).get("s5_complaint_dominant_check", {}).get("per_aspect", {})

rows = []
for j, a in enumerate(ASPECT_IDS):
    y, p = Y[:, j], P2[:, j]
    tp, fp, fn = int((y & p).sum()), int(((1 - y) & p).sum()), int((y & ~p).sum())
    prec = tp / (tp + fp) if tp + fp else np.nan
    rec = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * prec * rec / (prec + rec) if tp else (0.0 if y.sum() else np.nan)
    m = d3[d3["aspect"] == a]
    n3 = len(m)
    npos, nneg, nneu = int((m["y"] == 2).sum()), int((m["y"] == 0).sum()), int((m["y"] == 1).sum())
    acc = float(((m["pred"] == 0) == (m["y"] == 0)).mean()) if n3 else np.nan
    auc = roc_auc_score((m["y"] == 0).astype(int), m["p_neg"]) if n3 and 0 < (m["y"] == 0).sum() < n3 else np.nan
    prec_pos = int(((m["pred"] == 2) & (m["y"] == 2)).sum())
    pos_rec = prec_pos / npos if npos else np.nan
    flags = []
    if y.sum() < 15: flags.append("Task2: <15 gold positives")
    if not np.isnan(f1) and f1 < 0.70: flags.append("Task2: F1<0.70")
    if n3 < 30: flags.append("Task3: <30 pairs")
    if npos < 5: flags.append("Task3: <5 positive examples")
    if not np.isnan(acc) and acc < 0.75: flags.append("Task3: neg-vs-rest acc<0.75")
    v = s5.get(a, {}).get("verdict", "")
    if v: flags.append("S5: " + v)
    rows.append({"aspect": a, "T2_support": int(y.sum()), "T2_precision": round(prec, 2), "T2_recall": round(rec, 2), "T2_F1": round(f1, 2),
                 "T3_pairs": n3, "T3_neg": nneg, "T3_neu": nneu, "T3_pos": npos,
                 "T3_negVsRest_acc": round(acc, 2), "T3_AUC": round(auc, 2), "T3_pos_recall": round(pos_rec, 2),
                 "flags": "; ".join(flags)})
ev = pd.DataFrame(rows)
pd.set_option("display.width", 280); pd.set_option("display.max_colwidth", 120)
print(f"Task 2 at threshold {THR} (out-of-fold, applicability mask applied); Task 3 = grouped 5-fold OOF on gold+audit pairs")
print(ev.drop(columns="flags").to_string(index=False))
print("\n=== Flags ===")
for _, r in ev.iterrows():
    if r["flags"]:
        print(f"{r['aspect']:28s} {r['flags']}")
ev.to_csv(safe_path(f"{OUT_DIR}/aspect_evidence_table.csv"), index=False)
update_summary("aspect_evidence", {"table": ev.to_dict(orient="records"), "threshold": THR})
print("\nS9d done -> aspect_evidence_table.csv + bert_summary_v2.json['aspect_evidence']")

In [ ]:
# ===== S9b: Data quality and EDA summary per game (CPU only; needs S0; read-only except one csv + summary json) =====
import re
RAW_HINTS = {"royalmatch": ["royalmatch", "royal"], "genshin": ["genshin", "mihoyo"], "mlbb": ["mlbb", "legends"],
             "pubgm": ["pubg", "tencent"], "arknights": ["arknights", "yostar"]}

def parse_robust(s):
    try:
        t = pd.to_datetime(s, format="mixed", utc=True, errors="coerce")
    except (ValueError, TypeError):
        from dateutil import parser
        t = pd.to_datetime(s.map(lambda x: parser.parse(x) if isinstance(x, str) and x.strip() else pd.NaT),
                           errors="coerce", utc=True)
    return t.dt.tz_localize(None)

norm = lambda t: re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(t).lower())).strip()
def non_ascii_share(t):
    letters = [c for c in str(t) if c.isalpha()]
    return sum(ord(c) > 127 for c in letters) / len(letters) if letters else 0.0

def raw_rows(slug):
    """Best effort: unique reviewIds across raw files whose name matches the game (for the raw -> clean attrition)."""
    rd = f"{PROJECT_ROOT}/data/raw"
    if not os.path.isdir(rd):
        return None, []
    files = [f for f in os.listdir(rd) if f.lower().endswith(".csv") and any(h in f.lower() for h in RAW_HINTS[slug])]
    ids = set()
    for f in files:
        cols = pd.read_csv(f"{rd}/{f}", nrows=0).columns
        if "reviewId" in cols:
            ids |= set(pd.read_csv(f"{rd}/{f}", usecols=["reviewId"], dtype=str)["reviewId"].dropna())
    return (len(ids) if ids else None), files

out = {}
for slug in GAME_SLUGS:
    d = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv", dtype={"at": str})
    d["at"] = parse_robust(d["at"])
    n = len(d)
    text = d["content_clean"].fillna("").astype(str)
    wc = pd.to_numeric(d["word_count"], errors="coerce") if "word_count" in d.columns else text.str.split().str.len()
    nm = text.map(norm)
    dup_rows = nm.duplicated(keep=False)
    sents = pd.read_parquet(f"{PROJECT_ROOT}/outputs/predictions/{slug}_aspects.parquet", columns=["reviewId"])
    span = max((d["at"].max() - d["at"].min()).days, 1)
    ver = d["reviewCreatedVersion"].astype("string").str.split("_").str[0] if "reviewCreatedVersion" in d.columns else pd.Series([pd.NA] * n)
    mk = d["market"].value_counts() if "market" in d.columns else pd.Series(dtype=int)
    raw_n, raw_files = raw_rows(slug)
    m = {
        "reviews": n, "unique_reviewIds": int(d["reviewId"].nunique()), "sentences": len(sents),
        "sentences_per_review": round(len(sents) / n, 2),
        "markets": ", ".join(f"{k}:{v}" for k, v in mk.head(4).items()) or "n/a",
        "date_from": str(d["at"].min().date()), "date_to": str(d["at"].max().date()), "span_days": span,
        "reviews_per_day": round(n / span, 1),
        "version_missing_%": round(100 * ver.isna().mean(), 1), "n_versions": int(ver.nunique()),
        "top_version_share_%": round(100 * ver.value_counts(normalize=True).iloc[0], 1) if ver.notna().any() else np.nan,
        "words_median": float(wc.median()), "words_p90": float(wc.quantile(.9)), "words_p99": float(wc.quantile(.99)),
        "<=3_words_%": round(100 * (wc <= 3).mean(), 1), ">=100_words_%": round(100 * (wc >= 100).mean(), 1),
        "rows_with_duplicate_text_%": round(100 * dup_rows.mean(), 1), "unique_texts": int(nm.nunique()),
        "likely_non_english_% (>30% non-ASCII letters)": round(100 * (text.map(non_ascii_share) > 0.3).mean(), 1),
        "clean_differs_from_raw_%": round(100 * (d["content_raw"].fillna("").astype(str) != text).mean(), 1) if "content_raw" in d.columns else np.nan,
        **{f"stars_{k}_%": round(100 * (pd.to_numeric(d["score"], errors="coerce") == k).mean(), 1) for k in range(1, 6)},
        "weak_negative_%": round(100 * (d["overall_sentiment_weak"] == "negative").mean(), 1),
        "weak_neutral_%": round(100 * (d["overall_sentiment_weak"] == "neutral").mean(), 1),
        "weak_positive_%": round(100 * (d["overall_sentiment_weak"] == "positive").mean(), 1),
        "developer_reply_%": round(100 * d["replyContent"].notna().mean(), 1) if "replyContent" in d.columns else np.nan,
        "thumbs_up>0_%": round(100 * (pd.to_numeric(d["thumbsUpCount"], errors="coerce") > 0).mean(), 1) if "thumbsUpCount" in d.columns else np.nan,
        "flag_same_user_repeated": int(d["same_user_repeated_content"].sum()) if "same_user_repeated_content" in d.columns else "n/a",
        "flag_long_outlier": int(d["is_long_outlier"].sum()) if "is_long_outlier" in d.columns else "n/a",
        "raw_unique_reviewIds": raw_n if raw_n is not None else "not found",
        "raw_to_clean_retained_%": round(100 * n / raw_n, 1) if raw_n else "n/a",
    }
    out[slug] = m
    if raw_n is None:
        print(f"[{slug}] raw file(s) not matched automatically; files tried: {raw_files or 'none'}")

tab = pd.DataFrame(out)
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 100)
print(tab.to_string())
tab.T.to_csv(safe_path(f"{OUT_DIR}/data_quality_summary.csv"))
update_summary("data_quality", {k: {a: (None if (isinstance(b, float) and np.isnan(b)) else b) for a, b in v.items()} for k, v in out.items()})
print("\nS9b done -> data_quality_summary.csv + bert_summary_v2.json['data_quality']")

In [ ]:
# ===== S9f: Market composition audit (CPU only; needs S0; read-only except one csv) =====
# Why: the clean files pool several Google Play markets, and the market is tied to the collection run / timestamp format.
WINDOW_DAYS, N_BOOT = 14, 1000
TRANSITIONS = [("royalmatch", "37281.0"), ("royalmatch", "37743.0"), ("genshin", "6.7.0")]

def parse_robust(s):
    try:
        t = pd.to_datetime(s, format="mixed", utc=True, errors="coerce")
    except (ValueError, TypeError):
        from dateutil import parser
        t = pd.to_datetime(s.map(lambda x: parser.parse(x) if isinstance(x, str) and x.strip() else pd.NaT),
                           errors="coerce", utc=True)
    return t.dt.tz_localize(None)

frames = []
for slug in GAME_SLUGS:
    d = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv", dtype={"at": str},
                    usecols=["reviewId", "at", "score", "market", "overall_sentiment_weak"])
    d["at"] = parse_robust(d["at"]); d["game"] = slug
    ov = pd.read_parquet(f"{PROJECT_ROOT}/outputs/dashboard/overall_{slug}.parquet").drop_duplicates("reviewId")
    frames.append(d.merge(ov[["reviewId", "overall"]], on="reviewId", how="left"))
allr = pd.concat(frames, ignore_index=True)
allr["us"] = allr["market"].eq("us")
allr["pred_neg"] = (allr["overall"] == "negative").astype(float) * 100

# ---------- 1. Per game x market ----------
t1 = (allr.groupby(["game", "market"]).agg(n=("reviewId", "size"), first=("at", "min"), last=("at", "max"),
                                           avg_stars=("score", "mean"), weak_neg_pct=("overall_sentiment_weak", lambda s: 100 * (s == "negative").mean()),
                                           pred_neg_pct=("pred_neg", "mean")).reset_index())
t1["first"], t1["last"] = t1["first"].dt.date, t1["last"].dt.date
pd.set_option("display.width", 250)
print("=== 1. Reviews per game and market ===")
print(t1[t1["n"] >= 20].round(2).to_string(index=False))

# ---------- 2. Non-US share over time (14-day grid per game) ----------
print("\n=== 2. Share of non-US reviews (%) per 14-day window ===")
for slug in GAME_SLUGS:
    d = allr[allr["game"] == slug]
    if (~d["us"]).mean() < 0.01:
        print(f"{slug}: non-US share < 1% overall (not shown)")
        continue
    w = ((d["at"] - d["at"].min().normalize()).dt.days // WINDOW_DAYS)
    s = (100 * (~d["us"]).groupby(w).mean()).round(1)
    n = d.groupby(w).size()
    start = {k: (d["at"].min().normalize() + pd.Timedelta(days=int(k) * WINDOW_DAYS)).date() for k in s.index}
    print(f"{slug}: " + " | ".join(f"{start[k]}: {s[k]}% (n={n[k]})" for k in s.index))

# ---------- 3. Version-transition windows: market mix and US-only sensitivity ----------
cand = pd.read_csv(f"{OUT_DIR}/version_candidates_v2.csv")
rng = np.random.default_rng(42)
def boot_delta(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    da = rng.integers(0, len(a), (N_BOOT, len(a))); db = rng.integers(0, len(b), (N_BOOT, len(b)))
    d = np.nanmean(b[db], axis=1) - np.nanmean(a[da], axis=1)
    return np.percentile(d, 2.5), np.percentile(d, 97.5)
rows = []
for slug, ver in TRANSITIONS:
    c = cand[(cand["game"] == slug) & (cand["version"].astype(str) == ver)]
    t0 = pd.Timestamp(c["first_seen(p05)"].iloc[0])
    d = allr[allr["game"] == slug]
    pre = d[(d["at"] >= t0 - pd.Timedelta(days=WINDOW_DAYS)) & (d["at"] < t0)]
    post = d[(d["at"] >= t0) & (d["at"] < t0 + pd.Timedelta(days=WINDOW_DAYS))]
    for name, f in [("all markets", lambda x: x), ("US only", lambda x: x[x["us"]])]:
        a, b = f(pre), f(post)
        if len(a) < 30 or len(b) < 30:
            rows.append({"transition": f"{slug} {ver}", "subset": name, "n_pre": len(a), "n_post": len(b)}); continue
        lo1, hi1 = boot_delta(a["score"], b["score"]); lo2, hi2 = boot_delta(a["pred_neg"], b["pred_neg"])
        rows.append({"transition": f"{slug} {ver}", "subset": name, "n_pre": len(a), "n_post": len(b),
                     "nonUS_pre_%": round(100 * (~a["us"]).mean(), 1), "nonUS_post_%": round(100 * (~b["us"]).mean(), 1),
                     "stars_pre": round(a["score"].mean(), 2), "stars_post": round(b["score"].mean(), 2),
                     "stars_delta [95% CI]": f"{b['score'].mean() - a['score'].mean():+.2f} [{lo1:+.2f}, {hi1:+.2f}]",
                     "pred_neg_%_pre": round(a["pred_neg"].mean(), 1), "pred_neg_%_post": round(b["pred_neg"].mean(), 1),
                     "pred_neg_delta [95% CI]": f"{b['pred_neg'].mean() - a['pred_neg'].mean():+.1f} [{lo2:+.1f}, {hi2:+.1f}]"})
res = pd.DataFrame(rows)
print("\n=== 3. Version windows: all markets vs US only ===")
print(res.to_string(index=False))
t1.to_csv(safe_path(f"{OUT_DIR}/market_composition.csv"), index=False)
res.to_csv(safe_path(f"{OUT_DIR}/version_windows_market_sensitivity.csv"), index=False)